# Research Log: Framing the Game

Running notebook for brainstorms, hypotheses, experimental results, and reviewer-driven insights.

**Paper**: *Framing the Game: How Context Shapes LLM Decision-Making*  
**Authors**: Isaac Robinson, John Burden  
**Target**: ICLR 2026 resubmission  

---

## Key Reviewer Feedback (ICLR 2026 Round 1)

**Result**: Reject (scores: 4, 2, 6, 4)

### Must-address concerns
1. **Generalizability beyond one-shot PD** — all four reviewers flagged this. Need to either extend to other games or better justify the PD-only scope.
2. **LLM-as-judge QC without human validation** — meta-reviewer listed this as a primary rejection reason. Need human eval or at minimum judge-swap analysis.
3. **Descriptive, not mechanistic** — reviewers wanted to know *why* certain contexts drive cooperation, not just *that* they do (mVh3, aAtL).
4. **Limited reasoning model coverage** — mQG5 specifically flagged this; only 2 small R1 distillations tested.

### Addressable in resubmission
- Better topic selection to tell a clearer story
- 108-model registry now covers full reasoning model spectrum (o1, o3, R1, QwQ)
- Need behavioral/mechanistic analysis connecting topics to cooperation rates
- Need human validation study for vignette QC

---

## Brainstorms & Hypotheses

*(newest first)*

### 2026-02-20: Topic Redesign Brainstorm

**Problem**: Original 10 topics were redundant (5/10 politics, 2 generic) and produced results that were descriptive but not compelling. Reviewers asked "why" and we couldn't answer.

**Insight**: Topics should be chosen so the cooperation gradient is *predictable a priori* from domain norms, making the finding interpretable rather than just descriptive.

#### Promising axes of variation

1. **Moral valence of cooperation** — cooperation can be prosocial (sharing), neutral (trade), or antisocial (collusion, cover-up). If models cooperate less when cooperation is "wrong", that's evidence of learned moral reasoning overriding game theory.

2. **Cultural/geographic framing** — same scenario with different cultural context (Silicon Valley vs Tokyo vs Lagos). Would reveal training-data cultural stereotypes in strategic behavior. High bias/fairness relevance.

3. **Power asymmetry** — large corp vs startup, employer vs employee. Tests whose interests models default to.

4. **Observability** — private vs public negotiation. In one-shot PD, observability shouldn't matter. If it does, models are importing reputation-game logic.

5. **Stakes magnitude** — neighborhood dispute vs international crisis. Tests stake-sensitivity.

6. **Political dyads** — specific party matchups (R vs D, D vs Green, bipartisan committee). Tests whether models reflect real-world partisan polarization.

7. **Temporal/historical distance** — same dilemma in 1400 vs 1940 vs 2025. Models may grant "moral license" to defect in historical settings.

#### Open question
Which 2-3 axes to combine? The interaction effects (e.g., "cultural framing matters more in business than medicine") are where the paper gets strongest.

### 2026-02-20: Axes Selected [design]

**Selected 6 axes** (dropped stakes magnitude):
1. **Moral valence of cooperation** — prosocial vs neutral vs antisocial cooperation
2. **Cultural/geographic framing** — same scenario, different cultural context
3. **Power asymmetry** — symmetric vs asymmetric actor power
4. **Observability** — private vs public (shouldn't matter in one-shot PD but probably does)
5. **Political dyads** — specific party matchups with known real-world antagonism levels
6. **Temporal/historical distance** — same dilemma across time periods

**Next step**: Combine into a concrete topic set (10-13 topics) that covers multiple axes without being unwieldy. Key challenge is that 6 axes fully crossed would be enormous — need to pick topics that naturally embed 1-2 axes each.

### 2026-02-20: Power Analysis Results [result]

**Setup**: Two-proportion z-test, α=0.05, power=0.80, baseline coop ~55%

**Key takeaways**:
- To detect a **30pp difference** (e.g. moral valence extremes): only **~42 stories/group** needed — very cheap
- To detect a **15pp difference** (e.g. political dyads, temporal): **~170 stories/group** — moderate
- To detect a **10pp difference** (e.g. cultural framing): **~390 stories/group** — expensive
- To detect a **5pp difference**: ~1,550/group — probably not worth it

**Implications for axis selection**:
- **Moral valence** (expected 30pp effect): easiest to power, ~126 stories total. Do this.
- **Political dyads** (expected 15-25pp): ~250-700 total across 4 groups. Feasible.
- **Temporal distance** (expected 10-20pp): ~290-1,164 total. Feasible if effect is large enough.
- **Observability** (binary, expected 10-20pp): ~192-540 total. Cheap because only 2 groups.
- **Power asymmetry** (binary, expected 10-20pp): ~194-540 total. Same.
- **Cultural/geographic** (expected 5-15pp): **riskiest** — if effect is only 5-10pp, need 1,500+ stories/group. Could be underpowered unless effect is surprisingly large.

**Budget**: At $0.62/sweep for all 108 models, 200 stories/group × 10 groups = $1,240 total. Manageable.

### 2026-02-20: Budget approved [design]

**$1,200 budget approved** for full 108-model sweep at ~200 stories/group × ~10 groups. All 6 axes are a go including cultural/geographic (will accept risk of underpowering if effect is small).

**Next step**: Design the concrete topic set and config structure — which axes become topics vs config dimensions.

### 2026-02-20: Experimental Design Decisions [design]

**Structure**: topics × actor_types × observability × power_dynamic

**Decisions made**:
- **Drop `neutral` actor type** — keep only allies vs enemies for cleaner contrast
- **Drop `world_type`** (real vs imaginary) — wasn't a strong finding in v1, doubles cell count for little value
- **Add `observability`**: private vs public (binary)
- **Add `power_dynamic`**: symmetric vs asymmetric (binary)
- **Budget**: uncapped for now, prioritize at least 6 topics per axis category
- **Target**: ≥6 topics per axis (moral valence, political dyads, temporal, cultural/geographic)

**Cross product**: N topics × 2 actors × 2 observability × 2 power = N × 8 cells per topic

### 2026-02-20: External Brainstorm Synthesis [brainstorm]

**Key refinements from external model feedback:**

1. **Temporal axis**: Hold the *type* of interaction constant (e.g., "two powers negotiating a trade pact"), vary ONLY time period. Avoids confounding substance with era.
2. **Moral valence**: Ensure antisocial cooperation is explicitly illegal/exploitative, not just morally gray. Clean 3-bin structure (prosocial / neutral / antisocial).
3. **Political dyads**: Expand to include international adversaries (US vs China, US vs EU) for a fuller ideological-distance gradient. Monotonic prediction: cooperation ∝ perceived affinity.
4. **Cultural**: Structure by Hofstede-style dimensions (collectivist vs individualist, high-trust vs low-trust). Add Nordic (Stockholm) for high-trust baseline.
5. **Orthogonality**: Don't let moral valence confound actor_type axis. "Rival gangs" already implies enemies + antisocial — avoid this.
6. **Track refusals**: Model refusing to engage with antisocial cooperation is itself a finding.
7. **Explicit hypotheses**: H1 (moral valence), H2 (ideological distance), H3 (temporal modernity), H4 (cultural trust proxies).
8. **Optional 5th axis**: Anthropomorphism (individuals vs corporations vs AI systems vs governments) — interesting but parking for now.

### 2026-02-20: Temporal axis redesign — zoom into modernity [design]

**Insight**: The interesting question isn't "ancient vs modern" — it's whether models reflect shifting norms *within* recent history. Training data is 1000x denser for 2000s-2020s than for 1200 BCE. Cultural inflection points (social media, 2016 polarization, COVID) may produce measurable cooperation shifts even across single decades.

**New design**: 1 ancient anchor + dense modern granularity. Same underlying scenario: "two sovereign powers negotiating a trade agreement."

Proposed eras:
- ~1200 BCE (Bronze Age anchor — Hobbesian baseline)
- 1850s (Industrial, pre-world-wars, imperial competition)
- 1940s (WWII/postwar, institutional cooperation born out of catastrophe)
- 1970s (détente, Cold War thaw, pragmatic cooperation)
- early 2000s (post-9/11, "war on terror," multilateralism strained)
- 2010s (pre-Trump, peak globalization consensus)  
- 2020s (COVID, polarization peak, institutional distrust)
- near-future (speculative — tests whether models default to optimism or dystopia)

**Hypothesis refinement**: Cooperation may NOT be monotonically increasing with time. Possible non-linear pattern:
- 1940s bump (postwar institution-building)
- 2000s dip (post-9/11 unilateralism)
- 2020s dip (polarization, distrust)
- Near-future uncertain (utopian vs dystopian training data)

This non-linearity would be a much more interesting finding than a simple "modern = more cooperative" gradient.

### 2026-02-20: Temporal axis — tighter parallel structure [design]

**Problem**: Current temporal topics vary both the era AND the scenario type (grain trade, rail access, reconstruction aid, pandemic response). This confounds time with substance — if cooperation differs between "1940s reconstruction aid" and "2020s pandemic response," is that because of the era or because aid ≠ pandemic?

**Fix**: Use the exact same scenario template for all time periods. Only the date and period-appropriate nouns change. E.g., "two leaders negotiating a trade agreement in [era]" — same actors (political leaders), same action (trade negotiation), same stakes.

### 2026-02-21: Design rigor audit [design]

Reviewing the 30-topic design for confounds, missing controls, and methodological gaps before implementation.

#### Issues identified:

**1. Moral valence axis is confounded by domain**
"Hospitals sharing ventilators" vs "banks manipulating interest rates" differs in moral valence AND domain, actors, stakes, and familiarity. Any cooperation difference could be healthcare vs finance, not prosocial vs antisocial. 

**Fix**: Use matched pairs within the SAME domain where cooperation flips moral meaning:
- "Two pharma companies sharing drug trial data" (prosocial) vs "Two pharma companies coordinating drug pricing" (antisocial)
- Same actors, same industry — only the moral meaning of cooperation changes.

**2. No abstract baseline control**
Without a "naked PD" with no narrative framing, we can't measure how much ANY context shifts behavior. We can only say "X has more cooperation than Y," not "X increases cooperation by N pp over the rational baseline."

**3. Position counterbalancing**
If "cooperate" is always option A, models might show position bias. Need to randomize A/B assignment.

**4. Observability and power dimensions need concrete prompt specifications**
How exactly do these get injected into vignettes? Need standardized phrasing.

**5. No holdout set defined for the new design**

**6. Political dyads axis also varies scenario substance**
"Bipartisan Senate committee" vs "US vs China export controls" differs in both ideological distance AND institutional setting. Could hold setting constant: "negotiating a policy agreement" and vary only which parties/nations.

### 2026-02-21: Final topic set implemented [design]

All 6 methodological fixes from the rigor audit have been applied and implemented in `config.py`.

**Structure**: 43 topics (35 main + 8 holdout) across 5 axes, crossed with 3 binary dimensions.

#### Axes and topic counts

| Axis | Main | Holdout | Design principle |
|---|---|---|---|
| **Moral valence** | 12 (6 matched pairs) | 2 | Same domain, cooperation flips moral meaning |
| **Political dyads** | 8 | 2 | Same template ("negotiating a policy agreement"), vary only parties |
| **Temporal distance** | 8 | 2 | Same scenario ("two national leaders negotiating a trade agreement in [era]"), vary only era |
| **Cultural/geographic** | 6 | 2 | Same scenario ("two business executives negotiating a joint venture in [city]"), vary only location |
| **Baseline** | 1 | 0 | Abstract PD, no narrative framing |

#### Cross-cutting dimensions (280 total cells)
- **actor_type**: allies / enemies
- **observability**: private / public (with standardized prompt injection text)
- **power_dynamic**: symmetric / asymmetric (with standardized prompt injection text)

#### Methodological fixes applied
1. **Moral valence deconfounded** — matched pairs within same industry (pharma, tech, finance, agriculture, real estate, shipping)
2. **Abstract baseline added** — naked PD control for measuring effect of ANY narrative framing
3. **Political dyads parallel structure** — uniform template, only party names change
4. **Position counterbalancing** — noted for generator implementation (A/B swap already exists in analysis)
5. **Concrete dimension prompts** — `OBSERVABILITY` and `POWER_DYNAMIC` dicts have standardised phrasing
6. **Holdout set defined** — 2 topics per axis (8 total) for validation

#### Key hypotheses (testable)
- **H1**: Cooperation rate drops when cooperation = antisocial (moral valence)
- **H2**: Cooperation ∝ perceived ideological affinity (political dyads)
- **H3**: Non-linear temporal pattern — 1940s bump, 2000s/2020s dips (temporal)
- **H4**: Cultural trust proxies predict cooperation (Stockholm > Tokyo > Lagos) (cultural)

### 2026-02-21: Multi-game 2x2 extension implemented [design]

Directly addresses **ICLR reviewer concern #1** (generalizability beyond one-shot PD). Extended the framework from a single game to **four canonical 2x2 games**, each with distinct strategic structure:

| Game | Payoffs (AA,AB,BA,BB) | Nash | Labels |
|---|---|---|---|
| **Prisoner's Dilemma** | (3,3),(0,5),(5,0),(1,1) | BB | Cooperate / Defect |
| **Stag Hunt** | (4,4),(0,3),(3,0),(2,2) | AA, BB | Hunt Stag / Hunt Hare |
| **Chicken (Hawk-Dove)** | (3,3),(1,4),(4,1),(0,0) | AB, BA | Swerve / Dare |
| **Pure Coordination** | (2,2),(0,0),(0,0),(1,1) | AA, BB | Option Alpha / Option Beta |

**Key insight**: All 2x2 games share the same binary A/B decision structure. They differ only in payoff orderings and semantic meaning of "cooperate." Existing `PayoffMatrix`, `extract_decision`, label-swap logic, and agreement metrics all work unchanged.

#### Implementation summary
- **New module**: `games.py` with frozen `GameConfig` dataclass and `GAME_REGISTRY` (4 games)
- **`Story.game_type`** field added (default: `"prisoners_dilemma"` for backward compat)
- **Semantic decision labels** in prompts: e.g. `"Decision A (Hunt Stag)"` instead of bare `"Decision A"`
- **`game_type` as analysis dimension**: added to `_CATEGORIES` tuple, auto-propagates to chi-square, Cramer's V, entropy, predictive models
- **New visualization**: `plot_focal_rate_by_game()` — grouped bar chart of focal-decision rate per game per model
- **New presets**: `cross_game` (6 topics × 4 games × minimal dimensions) and `cross_game_full` (6 topics × 4 games × all dimensions)
- **Generalized game recognition**: `classify_game_recognition` now detects stag hunt, chicken, Nash equilibrium mentions, not just PD
- **155 tests pass** (up from 97), fully backward-compatible

#### New hypotheses enabled
- **H5**: Cooperation rate varies by game structure — PD (dominant-strategy defection) vs Stag Hunt (risk-dominance tension) vs Chicken (anti-coordination) vs Pure Coordination (focal point)
- **H6**: Context framing effects (moral valence, political dyads, etc.) interact with game type — e.g., moral valence may matter more in PD than in pure coordination
- **H7**: Models may show game-recognition effects differently across game types — recognizing "stag hunt" vs "prisoner's dilemma" may differentially affect cooperation

---

## Future Ideas

*(Cool directions beyond the current paper scope)*

### Simple games as training for complex games [brainstorm]

**Idea**: Use the 2x2 game framework as a **training curriculum** — fine-tune or few-shot LLMs on simple canonical games (PD, Stag Hunt, Chicken, Coordination) and measure whether performance transfers to more complex games like **poker**, **negotiation**, or **multi-player auctions**.

- Our framework already produces thousands of labeled (vignette, decision, payoff) tuples across 4 game types — this is a ready-made training set for strategic reasoning
- **Hypothesis**: Models trained on simple 2x2 games learn transferable strategic primitives (risk assessment, opponent modeling, payoff comparison) that improve play in complex incomplete-information games like poker
- **Why it might work**: Poker requires bluffing (Chicken-like), cooperation in multi-hand play (iterated PD), risk-dominant vs payoff-dominant reasoning (Stag Hunt), and coordination on betting conventions (Coordination) — all present in our 2x2 games
- **Why it might not**: Complex games have hidden information, sequential moves, and combinatorial action spaces that 2x2 games don't capture — the gap may be too large
- **Evaluation**: Compare fine-tuned models against baselines on poker bots (e.g. PokerRL benchmarks), multi-round negotiation tasks, or Diplomacy-style games
- **Cool angle**: If it works, it suggests LLMs can learn *abstract strategic reasoning* from simple contexts and generalize — a strong claim about emergent game-theoretic capability

### 2026-02-22: Added 3 new games to registry (now 7 total) [design]

Motivated by **literature review** of recent LLM game theory work. Registry now covers 7 canonical 2x2 games:

#### New games

- **Harmony (Prisoner's Delight)**: `(4,4),(2,3),(3,2),(1,1)` — R>T>S>P, cooperation is the **dominant strategy**. Serves as the critical **positive control** for PD: if context framing shifts cooperation in Harmony the same way as in PD, models are responding to framing, not game structure. Lorè & Heydari (2024, *Scientific Reports*) used this game for exactly this reason.
- **Battle of the Sexes**: `(3,2),(0,0),(0,0),(2,3)` — **asymmetric coordination** where both prefer to coordinate but disagree on which outcome. Tests a fundamentally different coordination challenge than Pure Coordination (which is symmetric).
- **Matching Pennies**: `(1,0),(0,1),(0,1),(1,0)` — **constant-sum** with no pure Nash equilibrium. Tests whether models can handle zero-sum opposition. Uses `(1,0)/(0,1)` representation to avoid negative payoffs in the happiness-framed prompts.

#### Literature context

- **TMGBench** (2024) covers all 144 Robinson-Goforth topology games but uses raw matrices, not contextualized vignettes — our contribution is the narrative framing dimension
- **Lorè & Heydari** (2024) tested PD, Stag Hunt, Snowdrift (≈Chicken), and Prisoner's Delight (≈Harmony) — we now cover all of these plus Battle of the Sexes and Matching Pennies

#### Updated hypotheses

- **H5 (updated)**: Harmony should show near-ceiling cooperation regardless of context (cooperation is dominant) — any context effect here is pure framing bias
- **H8**: Battle of the Sexes may reveal **asymmetric framing effects** — agent 1 vs agent 2 may respond differently to the same context since their payoff preferences diverge
- **H9**: Matching Pennies cooperation rate should be ~50% (random) if models reason correctly — systematic deviations reveal decision biases

**160 tests pass**, all presets updated to 7 games.

### 2026-05-05: Quality review of 504 PD stories (2026-05-05-sharp run) [result]

Ran `scripts/review_prisoners_dilemma.py` — programmatic checks on all 504 stories across 10 cells, plus API deep review (claude-sonnet-4.6 via OpenRouter) on 5 randomly sampled stories per cell (50 total, `random.seed(42)`).

#### Programmatic check results (all 504 stories)

| Criterion | Pass | Total | Rate | Status |
|---|---|---|---|---|
| **C1 Elicitation block integrity** | 504 | 504 | 100.0% | OK |
| **C2 No game-theory contamination** | 504 | 504 | 100.0% | OK |
| **C3 No outcome enumeration** | 504 | 504 | 100.0% | OK |
| **C4 Unresolved ending** | 490 | 504 | 97.2% | WARN |
| **C5 Context dimension fidelity** | 441 | 504 | 87.5% | FAIL* |

*C5 rate is heavily inflated by false positives in the name-list checker — see notes below.

#### API deep review results (50 sampled stories)

| Criterion | Pass | Total | Rate | Status |
|---|---|---|---|---|
| **C6a Temptation present** | 50 | 50 | 100.0% | OK |
| **C6b Enumeration free** | 26 | 50 | 52.0% | WARN |

#### Key findings and issues

1. **Outcome enumeration is a real generator defect (C6b, 52% pass rate)**. The programmatic C3 check passes 100% (which only detects pattern-matched multi-occurrence), but the API judge finds ~48% of stories explicitly walk through the four outcome combinations within the story body (e.g., "if she cooperates and he defects… if both defect…"). This is a genuine quality problem — explicit outcome enumeration in the vignette likely contaminates the decision elicitation by pre-framing the payoff structure. Worst cells: era__ancient (1/5), realism__realistic (1/5), contrast_domain__business (2/5), observability__private (2/5), observability__public (2/5).

2. **Gender fidelity checker has too many false positives (C5)**. The name list used for checking is incomplete — legitimately female names like Debra, Stephanie, Carolyn, Kathleen, Madison, Megan, Frances, Angela, Beverly, Kathy, Catherine, Virginia, etc. are all flagged as "possibly non-female." Real gender fidelity failures are likely much lower than the 38/50 flagged. Needs a more comprehensive name corpus (e.g., SSA baby names list).

3. **14 stories have resolved endings (C4)**. The "decided to" pattern fires even mid-story in some cases (describing a past decision by one of the agents, not the elicited decision). Need to narrow the check to the last ~300 chars of the body, not the whole text.

4. **False positive "orc" substring in realistic stories**. The fantasy keyword `"orc"` matches inside last names like "Okafor," "Orca," etc. — 6 stories in `realism__realistic` were wrongly flagged. Need to switch to whole-word matching.

5. **2 era__modern stories miss modern keywords**. One is about a flour shortage at a small bakery (legitimately contemporary but no tech/company vocabulary), one is about volunteer fire chiefs (radio/emergency setting). The keyword list is too tech-focused and misses non-tech modern settings.

6. **realism__fantasy has 54 stories instead of 50** — minor overrun, not a quality issue.

#### Overall assessment: **WARN**

The dataset is structurally sound (elicitation block, no GT terms, no programmatic enumeration all at 100%). The main actionable defect is **C6b outcome enumeration in the story body** — roughly half of stories walk through payoff combinations explicitly. This should be fixed in the generator prompt before the full production run.

### 2026-05-05: Paper analysis pipeline for sharp-narrative runs [design]

**Two unified XGBoost classifiers** (model_id as a feature, not per-LLM splits as in the original paper — we now have 7 frontier models):

1. **EmbeddingPredictor** — features = `[story_embedding (all-distilroberta-v1, 384d), model_one_hot]`; target = `cooperate` (1/0). Tests whether story text alone (conditioned on which LLM saw it) predicts the outcome.
2. **CategoricalPredictor** — features = `[model_one_hot, game_type, contrast_dim, gender, realism, era, contrast_domain, observability]`; target = `cooperate`. Tests whether the design knobs alone predict the outcome. Replaces the original `[topic, actor_type, observability, power_dynamic]` feature set, which is now near-zero-variance (held constant in the sharp design).

Both report **unified AUROC + per-model AUROC slice** on the held-out test set. 80/20 split, 5-fold CV grid search over the Appendix C grid.

**Swap-ablation** to measure A/B label-order bias:
- Re-eval the **first 5 stories from every one of the 70 cells** (7 games × 5 contrast dims × 2 levels) with `_swap_labels` applied
- 70 × 5 × 7 models = **2,450 extra calls**
- Full grid coverage at low depth; same budget as 7 cells × 50 stories but better coverage of the design space

**Pipeline gap to close**: existing `analysis/predictive.py` hardcodes old `(llama, claude, gpt4)` model names and the held-constant feature set. Need a new `analysis/loader.py` to assemble wide DataFrames from `data/runs/<id>/{stories,evals}/` JSONL, plus rewrite of `predictive.py` for the unified-classifier formulation, plus a `--phase swap` mode in the runner.

### 2026-05-05: Quality review of 501 stag_hunt stories (2026-05-05-sharp run) [result]

Ran `scripts/review_stag_hunt.py` — programmatic checks on all 501 stories across 10 cells, plus API deep review (`claude-sonnet-4.6` via OpenRouter) on 5 randomly sampled stories per cell (50 total, `random.seed(42)`).

#### Programmatic check results (all 501 stories)

| Criterion | Pass | Total | Rate | Status |
|---|---|---|---|---|
| **C1 Elicitation block integrity** | 501 | 501 | 100% | OK |
| **C2 No game-theory contamination** | 501 | 501 | 100% | OK |
| **C3 No outcome enumeration** | 501 | 501 | 100% | OK |
| **C4 Unresolved ending** | 501 | 501 | 100% | OK |
| **C5 Context dimension fidelity** | 501 | 501 | 100% | OK |

All programmatic checks pass cleanly. The reviewer script required significant keyword expansion vs. the PD reviewer:
- **Gender names**: confirmed dataset names (Beverly, Judith, Willie, etc.) were missing — added the full set directly
- **Fantasy fidelity**: ~20% of fantasy stories use unique world-building compounds (`wind-singer`, `cloud-fortress`, `seer-coven`) not in standard keyword lists — added regex patterns for `-singer`, `-caller`, `-fortress`, airship, kraken, leviathan, automaton, pirate, etc.
- **Realistic fidelity**: substring false positives (`elf`→"herself", `orc`→"force/porch") fixed with `\b` word-boundary matching; semantically ambiguous words (`quest`, `spell`, `dwarf`, `phoenix`, `tribe`) excluded from the realistic contamination check
- **Ancient era**: classical Mediterranean maritime vocabulary (trireme, Phoenician convoy, Red Sea monsoon, stone mole) added
- **Private observability**: stories express privacy through simultaneous-commitment mechanics ("each had to commit by 6pm", "if only one signed") rather than the word "private" — expanded with implicit coordination-blocking language
- **Business/political domains**: studio, NDA, publisher, pitch added for business; senator, caucus, whip, bill, amendment added for political

#### API deep review results (50 sampled stories)

| Criterion | Pass | Total | Rate | Status |
|---|---|---|---|---|
| **TRUST_TENSION_PRESENT** | 50 | 50 | 100% | OK |
| **ENUMERATION_FREE** | 34 | 50 | 68% | WARN |

#### ENUMERATION_FREE failures by cell

| Cell | Pass/n | Rate |
|---|---|---|
| contrast_domain__business | 4/5 | 80% |
| contrast_domain__political | 3/5 | 60% |
| era__ancient | 4/5 | 80% |
| era__modern | 4/5 | 80% |
| gender__female | 4/5 | 80% |
| **gender__male** | **2/5** | **40%** |
| observability__private | 3/5 | 60% |
| **observability__public** | **2/5** | **40%** |
| realism__fantasy | 4/5 | 80% |
| realism__realistic | 4/5 | 80% |

#### Key findings

1. **Trust tension is universally present (100%)** — every sampled story correctly frames the Stag Hunt core: Action A = high-reward coordinated path requiring mutual commitment, Action B = safe individual fallback. Strategic structure is right.

2. **Outcome enumeration is a persistent generator defect (32% of stories in sample)** — 16/50 stories explicitly walk through 2–3 payoff combinations in narrative prose (e.g., "If both monks sang, all Codices floated free. If only one sang, that monk lost their voice and the books did not move."). This is slightly better than the PD review (48% failing there) but still material.

3. **The stag_hunt game structure may inherently invite outcome narration** — the "both-must-commit-or-lose-all" mechanic is naturally explained by walking through the asymmetric scenarios. Generators seem to be setting up the tension by spelling out what happens in each case rather than conveying it through narrative implication alone. This is a generator system prompt issue.

4. **Programmatic C3 check is still too permissive** — regex-based check only catches multi-occurrence patterns; the API judge finds enumeration in single long paragraphs that describe the payoff structure conversationally.

#### Overall assessment: **WARN**

501 stories are structurally clean across all programmatic criteria. The **strategic framing is correct in every sampled story**. The primary actionable defect is **outcome enumeration in ~32% of stories** — the generator explains payoff combinations explicitly rather than implying them through narrative. Recommend tightening the generator system prompt with an explicit prohibition and "show don't tell" examples before the full production run.

### 2026-05-05: Quality review of ~1004 deadlock & harmony control stories (2026-05-05-sharp run) [result]

Ran `scripts/review_deadlock_harmony.py` — programmatic checks on all 1004 stories across 20 cells (10 deadlock + 10 harmony), plus API deep review (`anthropic/claude-sonnet-4.6` via OpenRouter) on 5 randomly sampled stories per cell (100 total, `random.seed(42)`).

#### Programmatic check results — DEADLOCK (501 stories, 10 cells)

| Criterion | Pass | Total | Rate | Status |
|---|---|---|---|---|
| **C1 Elicitation block integrity** | 501 | 501 | 100.0% | OK |
| **C2 No game-theory contamination** | 501 | 501 | 100.0% | OK |
| **C3 No outcome enumeration** | 501 | 501 | 100.0% | OK |
| **C4 Unresolved ending** | 494 | 501 | 98.6% | WARN |
| **C5 Context dimension fidelity** | 344 | 501 | 68.7% | FAIL* |

#### Programmatic check results — HARMONY (503 stories, 10 cells)

| Criterion | Pass | Total | Rate | Status |
|---|---|---|---|---|
| **C1 Elicitation block integrity** | 503 | 503 | 100.0% | OK |
| **C2 No game-theory contamination** | 503 | 503 | 100.0% | OK |
| **C3 No outcome enumeration** | 503 | 503 | 100.0% | OK |
| **C4 Unresolved ending** | 500 | 503 | 99.4% | WARN |
| **C5 Context dimension fidelity** | 307 | 503 | 61.0% | FAIL* |

*C5 failures are predominantly false positives in the keyword/name-list checker — see notes below.

#### API strategic faithfulness results (5 sampled per cell, 50 per game)

| Game | Pass | Total | Rate | Overall |
|---|---|---|---|---|
| **Deadlock** | 49 | 50 | 98.0% | **PASS** |
| **Harmony** | 25 | 50 | 50.0% | **FAIL** |

**Per-cell breakdown:**

| Cell | API | Status |
|---|---|---|
| deadlock__contrast_domain__business | 4/5 | WARN |
| deadlock__contrast_domain__political | 5/5 | OK |
| deadlock__era__ancient | 5/5 | OK |
| deadlock__era__modern | 5/5 | OK |
| deadlock__gender__female | 5/5 | OK |
| deadlock__gender__male | 5/5 | OK |
| deadlock__observability__private | 5/5 | OK |
| deadlock__observability__public | 5/5 | OK |
| deadlock__realism__fantasy | 5/5 | OK |
| deadlock__realism__realistic | 5/5 | OK |
| harmony__contrast_domain__business | 5/5 | OK |
| harmony__contrast_domain__political | 4/5 | WARN |
| **harmony__era__ancient** | **0/5** | **FAIL** |
| **harmony__era__modern** | **1/5** | **FAIL** |
| **harmony__gender__female** | **2/5** | **FAIL** |
| **harmony__gender__male** | **2/5** | **FAIL** |
| **harmony__observability__private** | **2/5** | **FAIL** |
| harmony__observability__public | 3/5 | WARN |
| harmony__realism__fantasy | 3/5 | WARN |
| harmony__realism__realistic | 3/5 | WARN |

#### Key findings

**1. Deadlock stories are high quality.** 98% API pass rate. The single failure (business cell story #47) had a character explicitly enumerating outcome combinations in dialogue. C4 failures (7 stories) are the same false-positive "decided to" pattern from the PD review — phrase appears in mid-story dialogue, not as an actual resolution of the elicited decision.

**2. Harmony has a systemic generation defect.** 50% API fail rate across 9 of 10 cells. The generator is injecting dramatic deliberation, temptation framing, and explicit outcome enumeration into what should be effortless, obvious decisions. Specific failure modes identified by the judge:
- **False tension**: stories describe "strange small weight," characters picking up/putting down a pen, "treacherous voice of glory," cold coffee, moaning wind — atmospheric framing that makes a dominant-strategy choice feel agonizing
- **Outcome enumeration**: explicit walk-through of "if A does X and B does Y" logic (e.g., "Holding out solo meant a worse deal regardless of what their partner did"), which violates the spirit of C3 even when the pattern-match check passes
- **Coordination-risk framing**: stories emphasize that a mismatch defaults to a worse outcome, implying strategic uncertainty around what should be trivially easy
- Worst cells: **era__ancient (0/5), era__modern (1/5)** — the ancient and period-drama settings especially trigger elaborate deliberation prose

**3. C5 context fidelity failures are overwhelmingly false positives.** Root causes:
- **Private observability keyword gaps** (~60–70% of all C5 failures): stories convey private scenarios through narrative context (isolated settings, sealed submissions, no communication channel) rather than using the explicit vocabulary the checker requires. Not a real story quality problem.
- **Fantasy keyword gaps**: stories use invented world-building terms (wyvern, sky-galleon, golem-engineer, kelpie-keeper, moonglass, coven) not in the checker's list. Real checker gap.
- **Ancient era keyword gaps**: period-accurate vocabulary (Mouseion, basilica, Piraeus, Lugdunum, publicani) not caught by the checker. Real checker gap.
- **Name list gaps**: legitimately gendered names flagged as unknown — Katherine, Marilyn, Gloria, Amber, Andrea, Kathy, Beverly, Judy, Christina, Kathryn, Megan, Maria, Kelly (female); Bryan, Keith, Bruce, Willie, Lawrence, Vincent, Jeremy (male). Checker issue, not story issue.

#### Overall assessment

- **Deadlock**: **PASS** (WARN classification driven entirely by checker false positives, not story defects)
- **Harmony**: **FAIL** — requires generator prompt revision before production use. The instruction to make Action A "effortless and obvious" is not being respected; the generator defaults to its PD-story register (tension, deliberation, temptation) even for dominant-strategy games.

#### Recommended fix for Harmony generator prompt

Add explicit negative constraints to the system prompt:
- Do NOT describe the agent deliberating or hesitating
- Do NOT mention the attractiveness of Action B or any temptation toward the alternative
- Do NOT use atmospheric cues (cold, dark, wind, silence) that create psychological weight
- Do NOT walk through what happens under each combination of choices
- The story should end with the agent moving naturally toward the obvious action, with no sense of internal conflict

### 2026-05-06: Refined mech interp goal — universal game structure representation [design]

The actual claim we want to make is:

> **The model maintains a universal representation of game structure — stories about the same game cluster together in activation space regardless of surface framing — and contextual framing lives in the residual variation around that structure.**

This is different from "where is the information processed" (the earlier probing framing). It's a claim about *representational geometry*.

---

#### The key experiment: cross-framing probe generalization

Train a `game_type` classifier using only stories from framing A (e.g. `gender__male`).  
Test it on stories from framing B (e.g. `gender__female`) — completely different character names, vocabulary, scenarios.

- **High test accuracy** → the game_type representation is *universal*: the model encodes "this is a Prisoner's Dilemma" in a way that doesn't depend on whether the characters are male or female, ancient or modern, business or political.
- **Low test accuracy** → the game representation is surface-specific (it's just picking up on vocabulary that happens to co-occur with each game).

**The contrast experiment**: train a `contrast_dim_level` classifier (male vs female) on PD stories, test on Stag Hunt stories. If context representations do NOT generalise cross-game (lower accuracy) while game representations DO generalise cross-framing, the asymmetry is the result.

**Output**: a generalization matrix — rows = train framing, columns = test framing, values = accuracy. If uniformly hot → universal. If diagonal → surface artifact.

---

#### The supporting experiment: RSA (Representational Similarity Analysis)

Compute pairwise cosine similarity between all story activations at each layer. Compare to two ground-truth similarity matrices:
- `S_game[i,j] = 1` if same game_type
- `S_context[i,j] = 1` if same contrast_dim_level

Compute Spearman correlation of the activation similarity matrix with each ground-truth matrix at every layer.

**Key finding to look for**: `r_game > r_context` at intermediate layers → game identity organises the representation more than framing → universal game structure. If the ordering flips in later layers (r_context rises), context becomes relatively more prominent near the decision.

---

#### What this answers

The paper sentence: *"We show that Gemma4 maintains a layer-specific universal representation of game structure: a linear classifier trained to identify game type from stories with one contextual framing transfers with [X%] accuracy to stories with entirely different framings. In contrast, a classifier trained to identify contextual framing from one game type transfers with only [Y%] accuracy to stories from a different game. This asymmetry suggests the model encodes game structure in a framing-invariant subspace while contextual variation modulates behavior in a game-dependent way."*

This directly answers reviewer concern #3 ("descriptive not mechanistic") without requiring a new behavioural experiment — it's a mechanistic characterization of the existing data.

---

#### Implementation

Added to `.worktrees/steering/game_theory_llm/steering/probing.py`:
- `cross_framing_probe(bundles, target_label="game_type", split_label="contrast_dim_level")` — the generalization matrix
- `game_rsa_all_layers(bundles)` — RSA r_game vs r_context per layer
- `run_full_probe_analysis(bundles)` — runs both + layer_probe + logit_attribution

Pipeline to run once stories are finalized:
```bash
python3 scripts/build_mech_interp_corpus.py \
    --src-dir data/runs/2026-05-05-sharp/stories \
    --out data/runs/mech-interp-v1/stories.jsonl --n-per-cell 50

python3 scripts/run_steering.py extract \
    --run-id mech-interp-v1 --stories data/runs/mech-interp-v1/stories.jsonl --split train
```

Then offline (local CPU):
```python
_, bundles = load_index_and_bundles("local_data/runs/mech-interp-v1")
results = run_full_probe_analysis(bundles)
# results["cross_framing_game"] → generalization matrix (game probe)
# results["cross_game_context"] → generalization matrix (context probe)
# results["rsa"] → r_game vs r_context per layer
```

### 2026-05-06: Mech interp design concern — within-cell variation vs. binary label signal [design]

**The problem**: Each cell (e.g., `gender__male`) contains 50 independently generated stories. They share the binary label (all have male characters) but vary enormously in scenario, setting, names, vocabulary. When we try to extract a "gender direction" via `mean(activations[male]) - mean(activations[female])`, we're averaging over a lot of within-cell noise. The direction we recover might just be "vocabulary typical of male-named stories" rather than anything abstract about how gender is represented.

The game-structure direction has this problem less severely — all PD stories share a consistently embedded payoff structure, so the PD/Harmony mean contrast is likely cleaner than the male/female contrast.

This asymmetry could make the game probe look artificially stronger than the context probe even if both are equally "real."

---

#### Three ways to handle it

**Option A — Let it average out (easiest, probably fine)**

With 50 stories per level, the within-cell noise should average out and the mean direction will capture whatever is systematically shared across all male stories (i.e., male names, male pronouns, masculine-coded vocabulary). This is still a real representation — it just might be surface-level rather than abstract.

**Diagnostic check before committing**: compare CategoricalPredictor AUROC vs EmbeddingPredictor AUROC on the behavioral data. If they're close, the binary labels explain most of the variance → averaging-out works. If EmbeddingPredictor >> CategoricalPredictor, within-story variation is dominant → need more control.

**Option B — Residual probing (controls for surface text, isolates abstract representation)**

Instead of probing raw Gemma4 activations, probe the *residual* after regressing out the story's sentence embedding (all-distilroberta-v1):

```
residual_activation = gemma4_activation - projection_onto(sentence_embedding_subspace)
```

Then train game and context probes on the residual. This isolates what Gemma4 has computed *beyond* what's already in the surface text — a genuinely internal representation rather than a reflection of vocabulary.

If context probes are strong on raw activations but weak on residuals, the effect is surface-level (vocabulary, names). If context probes remain strong on residuals, Gemma4 is constructing an abstract representation of the context dimension.

This is probably the right thing to do and costs almost nothing extra (we're already computing sentence embeddings for EmbeddingPredictor).

**Option C — Generate a matched-pairs mech interp subset (cleanest, requires new generation)**

For each contrast dim, generate 20–30 "minimal pairs": identical stories where only the binary manipulation is swapped. Same characters, same scenario, same setting — just swap he→she, ancient→modern, private→public, etc.

This is standard in mech interp (e.g., ROME, IOI studies use minimal pairs) and gives the cleanest possible direction estimate. The downside: requires new story generation with a template-based approach rather than free generation.

**Practical version**: don't regenerate from scratch. Take existing stories from the `male` cell and do a simple find-replace to get `female` versions. This is imperfect (doesn't adjust everything) but produces clean activation pairs for the directional analysis.

---

#### Recommendation

Do **Option B** (residual probing) as the default analysis — it's free and controls for the obvious confound. Use it to test whether the context representation is surface-level or abstract.

Add **Option C** (minimal pairs) as a supplement for the gender dimension specifically, since find-replace is trivial and gender has the most surface confounders (names, pronouns). If the gender direction from minimal pairs matches the direction from the full-population mean contrast, it validates the averaging-out approach.

**Option A** is the fallback if Options B and C produce similar results — in that case, just report the simpler analysis.

---

#### Why the within-cell variation is also an opportunity

The stories within a cell vary in ways that aren't prespecified — different industries, settings, plot structures. This variation predicts cooperation too (hence EmbeddingPredictor >> baseline). We could analyze this "unspecified framing variation" directly:

- Run PCA on the story embeddings within each cell
- The top PC captures the dominant axis of within-cell variation
- Check if this PC predicts cooperation (above the binary label)
- If it does, probe Gemma4 activations for this PC direction — it's a data-driven framing dimension we didn't pre-specify

This would be a bonus finding: "beyond our prespecified contrast dimensions, models respond to latent narrative variation in ways that are predictable from story embeddings and recoverable from internal activations."

### 2026-05-06: Mechanistic interpretability experiments — separating game structure from context in representation space [brainstorm]

The core question: does the model *have* a clean internal representation of game structure that gets overridden by context, or are game structure and context entangled from the start?

We have Gemma4 (27B) available on Modal with weight access. The 7-game × 5-contrast-dim × 2-level design gives us an orthogonal factorial structure that most mech interp papers don't have — we can vary game and context independently, which makes the decomposition clean.

---

#### Experiment 1 — Layer-by-layer linear probing (the foundation)

For each transformer layer l, extract the **residual stream activation** at the position of the final story token (or first decision token) for every story in the dataset.

Train two linear classifiers at each layer:
- **Game probe**: predict `game_type` (7 classes) from the activation
- **Context probe**: predict `contrast_dim_level` (e.g., male/female, ancient/modern) from the activation

Plot probe accuracy vs. layer for both.

**What to look for:**
- If game probe peaks early and decays → game structure is computed but then written over
- If context probe peaks late (near output) → context is the proximate cause of the decision
- If both are high at the decision layer → both are present; need causal tracing to say which one *drives* behavior
- If game probe is LOW at all layers → the model never cleanly separates game structure from context; they're entangled from embedding time

**Data needed**: 350 stories (5 per cell × 70 cells) — we already have these from the pilot run.

**What's needed from Modal**: forward pass with residual stream hooks at each layer. Store activations one layer at a time to manage 27B-param memory. Token position = last token before the elicitation block, or first generated A/B token.

---

#### Experiment 2 — Logit attribution (which representation drives the final choice?)

Decompose the final `logit(A) - logit(B)` into contributions from each layer's residual stream update using the **logit lens / direct logit attribution** technique:

```
logit_diff = W_U @ (sum over layers of delta_h_l)
```

where `W_U` is the unembedding matrix projected onto the A-vs-B direction and `delta_h_l` is the contribution of layer l to the residual stream.

This tells you which layers are responsible for the final preference for A vs. B — without having to do causal interventions.

**Cross with probing results**: if layer l has high game probe accuracy but low logit attribution, it means the model *has* game structure information at that layer but it's not what's driving the decision. If layer l has high context probe accuracy AND high logit attribution, context is causally relevant.

**This is the single most compelling figure for the paper**: a bar chart showing logit attribution per layer, with game-type probe accuracy and context probe accuracy overlaid. If the high-attribution layers are also the high-context-probe layers (not the high-game-probe layers), the story is tight.

---

#### Experiment 3 — Activation patching / causal tracing

The classic Meng et al. / Elhage et al. approach: **patch activations from story A into story B** at specific layers, measure whether the model's decision changes.

**Design A — same game, different context (context patch)**:
- Source: `game=PD, context=ancient` → model cooperates
- Target: `game=PD, context=modern` → model defects
- Patch source activations into target at each layer; measure cooperation probability
- The layer(s) where patching flips the decision = the layer(s) that carry causally relevant context information

**Design B — same context, different game (game structure patch)**:
- Source: `game=Harmony, context=ancient` → model cooperates (because dominant strategy)
- Target: `game=PD, context=ancient` → model defects (rational)
- Patch source activations: does injecting Harmony activations into a PD story push the model toward cooperation?
- This directly tests whether game structure representations are causally active

**Key prediction**: Design A patches should flip decisions at later layers (context is processed late); Design B patches should have less consistent effects if the model is primarily driven by narrative context rather than game structure.

---

#### Experiment 4 — Orthogonal subspace decomposition

Use **contrastive pairs** to extract the principal directions for game structure and context:

```python
# Game direction: same context, different game
game_direction = mean(activations[game=PD]) - mean(activations[game=Harmony])

# Context direction: same game, different context  
context_direction = mean(activations[context=male]) - mean(activations[context=female])
```

Measure the **cosine similarity** between `game_direction` and `context_direction`:
- High similarity → entangled representations (game and context processed by same circuits)
- Near-zero → orthogonal representations (model has cleanly separated them)

Then project individual story activations onto each direction and measure how well each projection predicts the model's decision (A vs. B). This is a continuous version of the probing experiment.

**The intervention version**: add `game_direction` or `context_direction` as a **steering vector** to the residual stream at inference time and measure the effect on cooperation rate. This is **representation engineering** and directly tests causal relevance of each direction.

---

#### Experiment 5 — SAE feature attribution (if Gemma4 SAE is available)

If Gemma4 has open-source sparse autoencoders (Gemma Scope covers Gemma 2 series), decompose residual stream activations into interpretable features.

- Identify features that fire specifically on game-structural content (payoff structure, Nash, optimal, defect/cooperate in strategic sense)
- Identify features that fire on contextual content (setting, era, character names, domain vocabulary)
- Measure which features contribute most to `logit(A) - logit(B)` via attribution

This gives interpretable, human-readable explanations for *why* specific framings drive specific decisions — the most compelling form of mechanistic evidence for a paper audience.

---

#### The paper narrative this enables

Current claim: "contextual framing affects cooperation rates across 7 games and 5 contrast dimensions."

With mech interp: "We provide direct mechanistic evidence for *why* framing matters. Using linear probing and logit attribution on Gemma4-27B, we show that (1) the model encodes both game structure and contextual framing as linearly separable directions in the residual stream; (2) contextual framing directions have substantially higher logit attribution to the final A/B decision than game structure directions; (3) causally patching context representations across stories changes decisions at a higher rate than patching game-structure representations. Together, these results suggest that LLMs prioritize narrative context over strategic structure at the decision layer, even when the game structure is represented and accessible."

---

#### Practical prioritization

| Experiment | Effort | Payoff | Do? |
|---|---|---|---|
| E1 Layer probing | Medium (hook setup, ~350 forward passes) | High (publishable on its own) | **Yes, do first** |
| E2 Logit attribution | Low once E1 is running (same activations) | Very high (ties probing to causation) | **Yes, same pass as E1** |
| E3 Activation patching | Medium-high (N² pairs) | Very high (cleanest causal claim) | **Yes, top priority** |
| E4 Subspace decomposition | Low once E1 is running | High (compelling visualization) | **Yes, cheaply extends E1** |
| E5 SAE attribution | High (requires SAE availability check + new setup) | High but speculative | **Maybe, pending SAE availability** |

E1 + E2 + E4 can be done in a single Modal job (same forward pass, extract activations, run probing offline). E3 requires a second set of forward passes with patching. Total Modal compute: probably ~2-4 GPU hours on A100.

### 2026-05-06: Why does framing matter? Mechanistic hypotheses [brainstorm]

Reviewer concern #3 is "descriptive not mechanistic." We can address this without new experiments — the 7-game design already contains several sharp mechanistic tests. The key is to frame existing analyses as *tests of competing mechanisms* rather than descriptive statistics.

---

#### The four candidate mechanisms

**M1 — Narrative schema substitution**
The model replaces "what is game-theoretically optimal?" with "what do people in situations like this typically do?" When the scenario invokes a recognizable social script (rivals, allies, medieval traders, price-fixing executives), the model retrieves training-data base rates for that script rather than computing over payoffs.

- **Prediction**: cooperation rate should correlate with the "expected" social outcome for the scenario type, regardless of game structure
- **Test**: compare framing effects in PD (where schema→coop conflicts with game theory→defect) vs Harmony (where schema→coop aligns with game theory→coop). Harmony strips out game-theoretic ambiguity — any framing effect there is *pure schema*, not strategic reasoning.
- **Verdict strength**: if Harmony shows framing effects of similar magnitude to PD, M1 is confirmed

**M2 — RLHF prosocial bias × moral valence**
RLHF instills a strong prior toward prosocial behavior. When "cooperation" in the story is prosocial (sharing vaccines, research data), RLHF-trained models get a push toward A. When cooperation is antisocial (price-fixing, collusion), they get a push away. Base models without heavy RLHF should show smaller moral valence effects.

- **Prediction**: moral valence effect size (Cramer's V) should correlate with degree of RLHF / instruction tuning across models
- **Test**: compare base models vs RLHF/instruction-tuned variants in our 108-model registry for the moral valence cells specifically (we have matched pairs within the same domain)
- **Interesting corollary**: refusal rates on antisocial-cooperation cells are also informative — a model that refuses to engage is the *extreme* expression of RLHF prosocial bias

**M3 — Narrative equilibrium selection**
In games with multiple Nash equilibria (Stag Hunt, Battle of Sexes, Coordination), there is no single "correct" answer — both players coordinating on either equilibrium is rational. Framing is doing real epistemic work here: it provides a **focal point** that rational players use to coordinate. This is not a failure of game-theoretic reasoning — it's how humans actually solve coordination problems.

- **Prediction**: framing effect sizes should be *larger* in multi-equilibrium games (Stag Hunt, Battle of Sexes, Coordination) than in dominant-strategy games (PD, Deadlock, Harmony)
- **If confirmed**: we can frame this positively — "LLMs use context to resolve equilibrium indeterminacy the way humans do" — rather than negatively ("framing overrides game theory"). This is a much stronger paper.
- **The cross-game effect size comparison is the key figure**: plot Cramer's V for framing effects per game type. If the predicted ordering holds (multi-eq > dominant-strategy), M3 is supported.

**M4 — Iterated-game logic leakage**
Observability should not affect optimal play in one-shot games (defection is dominant in PD regardless of whether others can see). But in *repeated* games, public observability creates reputation incentives. If models show cooperation boosts under public observability, they are importing iterated-game logic into a one-shot framing — a specific, falsifiable claim.

- **Prediction**: observability effect should be larger than any other contrast dim, especially in PD (where it most conflicts with game theory)
- **Counter-prediction**: if observability effects are *not* present, models may actually be doing one-shot reasoning correctly and the other framing effects are the anomaly

---

#### The mechanistic test matrix

| Mechanism | Sharpest test in current design | Expected finding |
|---|---|---|
| **M1 Narrative schema** | Framing effect in Harmony | Effect present and large → M1 confirmed |
| **M2 RLHF moral valence** | Base vs instruct models on moral valence cells | Effect larger in instruct → M2 confirmed |
| **M3 Equilibrium selection** | Effect size: multi-eq vs dominant-strategy games | Cramer's V higher in Stag Hunt/BotS → M3 confirmed |
| **M4 Iterated-game leakage** | Observability effect size vs other dims | Observability largest in PD → M4 confirmed |

---

#### The linear probing angle (mechanistic, requires Gemma4 activations)

The deepest mechanistic test would be to ask: at the point where the model generates its A/B decision, has the **game structure representation** been suppressed by the **context representation**?

- Train a linear probe to predict `game_type` from residual stream activations at each layer
- Train a separate probe to predict `contrast_dim_level` (e.g., male/female, ancient/modern) from the same
- **Hypothesis**: context probe accuracy at the decision layer exceeds game-structure probe accuracy → context has displaced game computation in the final representation
- **Bonus**: probe accuracy by layer tells you *when* context "takes over" — if game structure representation peaks at layer 10 and context peaks at layer 28, framing is acting late in the processing pipeline, which is consistent with it operating as a final override rather than a modulation of early computation

This is the angle that lets us write something like: "at the layer responsible for the final decision, the model's representation of the scenario's contextual framing has higher predictive validity than its representation of the underlying game structure."

---

#### How to frame the paper argument

The current paper says: *framing affects cooperation*. Reviewers want: *why?*

The answer we can support with existing data: **LLMs use narrative context as a focal point for equilibrium selection, but this heuristic leaks into dominant-strategy games where it has no game-theoretic justification.** Specifically:
- In Stag Hunt and Battle of Sexes: framing effects are *rational* (they resolve genuine strategic uncertainty)
- In PD and Harmony: framing effects are *irrational* (dominant strategy is clear; context should be irrelevant)
- The fact that effect sizes are similar across both classes is the finding: LLMs apply narrative reasoning uniformly, regardless of whether the game warrants it

This reframes the paper from "LLMs are irrational" to "LLMs apply a context-heuristic that works well in coordination games but transfers inappropriately to dominance games" — a cleaner and more interesting claim.

### 2026-05-06: Harmony false-tension fix completed across all 10 files [result]

Ran `scripts/fix_harmony_tension.py` (with per-file subprocess isolation) to detect and regenerate Harmony stories with false strategic tension in `data/runs/2026-05-05-sharp/stories/`.

#### Detection and fix summary

| File | Stories | Bad detected | Fixed | Notes |
|---|---|---|---|---|
| `contrast_domain__business` | — | — | — | Already fixed in prior session |
| `contrast_domain__political` | — | — | — | Already fixed in prior session |
| `era__ancient` | 50 | 1 | 1/1 | Prior run falsely flagged 32 (over-triggering) |
| `era__modern` | 50 | 3 | 3/3 | |
| `gender__female` | 50 | 0 | — | Already clean |
| `gender__male` | 50 | 2 | 2/2 | |
| `observability__private` | 50 | 3 | 3/3 | 1 story needed 2 attempts (enum on attempt 1) |
| `observability__public` | 50 | 0 | — | Already clean |
| `realism__fantasy` | 51 | 1 | 1/1 | |
| `realism__realistic` | 50 | 4 | 4/4 | |
| **Total** | **501** | **14** | **14/14** | All 10 files report OK |

#### Key design decisions that worked

- **Detector**: `claude-sonnet-4.6` with a prompt distinguishing **atmospheric setting** (OK) from **decision-level tension** (bad). The prompt was refined to prevent false-positives on literary atmosphere (lamps, soldiers, gravitas) that is legitimate in ancient era stories.
- **Regenerator**: `claude-opus-4.7` via `generate_batch()` with `FIXED_HINT` — a strongly-worded framing constraint prohibiting hesitation, dread, outcome enumeration, and "courage needed" framing.
- **Acceptance gate**: **enum-only** (objective regex patterns: `if only one`, `if neither`, sequential combos). Re-validation with the tension detector was intentionally removed.

#### Critical insight: enum-only gate is the right call

An earlier run (prior session) applied the full detector as a re-validation gate on regenerated ancient stories. The detector flagged all regenerated stories as "still-bad" — not because they had decision-level tension, but because the ancient era's inherent literary gravitas (period-accurate atmosphere, weighty settings) consistently triggered the detector even after the FIXED_HINT was applied.

Removing re-validation and trusting the FIXED_HINT was validated by the data: business and political (fixed in prior session) proved the FIXED_HINT reliably produces tension-free stories. The ancient file in this session found only 1 genuinely bad story (vs. 32 falsely flagged before), confirming the detector over-triggers on ancient era atmosphere.

**The lesson**: objective, pattern-based gates (enumeration regex) are more robust than LLM-judge re-validation for acceptance criteria, especially when the judge's signal is confounded by surface features (era vocabulary, literary atmosphere) that are orthogonal to the quality criterion being enforced.

### 2026-06-07: Tier-1 scaled RLVR (Qwen3-30B-A3B) operation-overlap test — NULL at scale [result]

**Question**: Does the probe-scale (Qwen3-4B) game-RLVR reasoning-transfer effect survive to a strong base, and is it **operation-specific** (H_op: improvement scales with trained-operation overlap) or **general output discipline** (H_gen: uniform lift)?

**Setup**: GRPO via Tinker cookbook on Qwen3-30B-A3B-Instruct-2507. Trained on 4 game families (`level_k`, `iterated_dominance`, `bargaining`, `subtraction_game`), depths 2–6, 1,200 prompts, 38 batches. Pre-registered trained-op tags `{backward_induction, nested_belief, iterated_elimination, modular_combinatorial}`. Eval over a 9-benchmark suite; measurability filter keeps (benchmark,depth) cells with base accuracy in [0.10, 0.90].

**Training was healthy**: reward −0.034 → ~0.10 smoothed, KL ≈ 0.0014 (controlled), entropy ≈ 0.34 (stable). Checkpoint `tinker://2cb36c73…:train:0/sampler_weights/final`.

#### Verdict: **NULL** — no transfer, and the in-domain anchor fails to replicate.

| benchmark | base | rlvr | Δ raw | Δ acc-among-parsed | overlap |
|---|---|---|---|---|---|
| **depth_extrap** (in-domain, depths 7–8) | 0.475 | 0.431 | **−0.044** | −0.044 (real) | ov=1.0 |
| dyck | 0.489 | 0.300 | −0.189 | **−0.021 (≈flat)** | ov=0 |
| boolean_eval | 0.361 | 0.278 | −0.083 | flat (net) | ov=0 |
| mmlu_pro | 0.693 | 0.607 | −0.087 | −0.040 (half drift) | ov=0 |
| gsm8k (control) | 0.925 | 0.930 | **+0.005** | clean ✓ | — |

- **H_op not supported**: overlap slope **+0.020, p=0.584** (n.s.) in `dimp ~ ov + depth + C(benchmark)`.
- **H_gen not supported**: intercept +0.015 but every per-benchmark Δ ≤ 0.
- **Depth-extrapolation FAILS at scale**: 4B gave **+12.5 pp (p≈0.02)**; 30B gives **−4.4 pp**.
- **No-regression control clean**: gsm8k flat/positive, parse-rate up — GRPO did not damage core ability.
- **Most held-out "regressions" are parse-rate/format drift**, not reasoning loss: dyck acc-among-parsed is flat (0.418→0.397), ~half of mmlu_pro's drop is parse (0.807→0.740). The RLVR model shifted toward the bare-`<answer>` game format, mildly hurting MC/format-sensitive parsing.

#### Key confound: **headroom exhaustion (curriculum ceiling)**

This is **not** a clean refutation of scale-transfer. The trained game depths (2–6) are **at ceiling for the 30B base** (`prontoqa`/`knights_knaves` = 1.00, `ordering` = 0.97; `depth_extrap` only became *measurable* at depths 7–8). Reward only reached ~0.10 because most rollouts were already correct → little group-relative signal. GRPO had almost nothing to teach, and the small policy shift it induced perturbed format without adding reasoning skill.

**Mechanistic reading**: the game-RLVR transfer effect is **capability-bounded** — it appears when the base has headroom on the trained operations (true at 4B, false at 30B for depths 2–6).

#### Go / no-go

Ran the 30B-A3B point per the pre-registered decision tree → null. **Clean follow-up = Tier-1b**: re-train on **deeper games (depths 6–10)** where the 30B base has headroom, then re-test depth-extrapolation (depths 11–13) + held-out suite. If transfer reappears → effect is real but **curriculum-gated**; if still null → effect is genuinely **capability-bounded to small models**.

Result doc: `docs/results/scaled_rlvr_tier1_result.md`. Artifacts: `data/runs/gt_rlvr/{overlap_analysis.json, tier1_30b/grpo_curve.png, t1_base_*.json, t1_rlvr_*.json}`.

### 2026-06-07: Tier-1b deep game-tree curriculum (Qwen3-30B-A3B) — in-domain skill REAL, transfer NOT [result]

Tier-1 was **headroom-confounded** (trained on *saturating* freetext families — level_k decays to 0, bargaining converges, iterated_dominance/subtraction are one-step — so "deeper" wasn't "harder" and the 30B was at ceiling). Tier-1b fixes this by training on **game-tree minimax backward induction**, which genuinely scales (2^depth leaves, non-saturating) and is the family that gave +12.5pp at 4B.

**Headroom screen** confirmed a sharp capability cliff on the 30B base (real headroom): d3=1.00, d4=0.55, d5=0.15, d6=0.00, d7=0.00. So curriculum = **d3→4→5** (train where GRPO has within-group reward variance); depth-extrapolation test = **d6–7** (base floored).

**Training healthy & curriculum-shaped**: reward plateaus track depth bands (d3≈0.97 frac_all_good 0.72 → d4 climbs 0.58→0.69 frac_mixed 1.00 → d5≈−0.05 frac_all_bad ~0.35); entropy steps up; KL controlled. Far healthier than Tier-1 (which had no gradient at all).

**Before → after**:
- **In-domain (d3–5): 0.611 → 0.711 (+10pp)** — driven by **d4 +23.3pp (0.633→0.867)** and **d5 +10pp (0.20→0.30)**. Genuine operation learning — the thing Tier-1 never achieved.
- **Depth-extrap (d6–7): 0.000 → 0.006** — NO extrapolation; floored. parse_rate ~0.04 = a **generation-length wall** (can't fit backward induction over 64–128 leaves in 3072 tok). **4B +12.5pp does NOT replicate at 30B.**
- **Cross-task transfer (measurable held-out)**: boolean +2.5pp, mmlu_pro +2.7pp, dyck −1.5pp → mean **+1.2pp**, all sub-3pp. **Fails the ≥3-benchmark rule.**
- **gsm8k control: 0.930 → 0.925** (flat) — no regression. ✓

**Verdict (Tier-1 + Tier-1b)**: game-structured RLVR produces **genuine in-domain skill acquisition but not generalizable reasoning transfer at 30B** — neither depth-extrapolation nor cross-task. The gain is real (+23pp at the trained depth) but task- and depth-bounded. Answers the driving question: at scale, the game structure makes the model better **at the trained game**, and the skill stays local; the 4B signals were small-model in-distribution effects.

**Open follow-ups (would refine, not overturn)**: (a) depth-extrap retest at 6–8k tokens to separate generation-wall from reasoning-wall; (b) multi-epoch/larger frontier curriculum; (c) mixed-operation deep curriculum for cross-op transfer.

Doc: `docs/results/scaled_rlvr_tier1b_result.md`. Artifacts: `data/runs/gt_rlvr/{overlap_analysis_tier1b.json, tier1b_30b/grpo_curve.png, t1b_base_*.json, t1b_rlvr_*.json, screen_gametree_b2.json}`.

### 2026-06-07: Tier-1b token retest + domain expansion [result]

**(1) Depth-extrapolation floor was a TOKEN-WALL artifact — but the clean Δ is still zero.** Re-ran gametree d6 at 8192 tokens (vs 3072): base parse 0.06→0.90, base acc **0.000→0.212** (the 30B base CAN do ~21% of depth-6 trees given room). With the wall removed, the unconfounded depth-extrapolation test: **base d6=0.212, rlvr d6=0.212, Δ=+0.000**. The +23pp in-domain d4 gain does NOT carry even one step OOD to d6 — flat (no lift, no regression). So the 4B +12.5pp does not replicate at 30B, now confirmed as genuine generalization failure, not truncation. (rlvr parse 0.83<base 0.90: RLVR trained under 2048-tok cap → slightly shorter CoT; acc-among-parsed identical.) Lesson: **always check parse_rate before calling a depth "floored."**

**(2) Domain expansion — 3 new reasoning-operation generators** (structure-first, exact verifier, knowledge-free, depth-scaling, non-saturating; 15 tests pass):
- `nim_grundy` (**combinatorial_game**) — multi-heap Nim / Sprague-Grundy XOR; answer = winning move size from heap 1.
- `opponent_id` (**inductive_rule**) — infer a finite-memory (2^k table) opponent rule from a de Bruijn-covered transcript, predict #cooperations over a fresh continuation. *Best answer diversity* (modal-guess 0.25–0.38 all depths).
- `signal_abduce` (**abductive_inference**) — observe an action, infer the unique type that rationalizes it (payoffs regenerated until type→signal is a bijection ⇒ unique abduction). Good from d3+.

Reasoning taxonomy × game structure: the deduction-only suite (backward induction, IESDS, nested belief, constraint-sat) now gains inductive/abductive/combinatorial. Remaining gaps: epistemic (muddy-children/common-knowledge) and probabilistic/EV (mixed-NE value, Bayesian posterior).

**Why it matters**: Tier-1b proved depth-in-ONE-operation stays local. The untested hypothesis is **breadth** — a diverse multi-operation curriculum is the better bet for transfer to held-out general reasoning (BBH-Hard, ZebraLogic). That's the next experiment.

Doc updated: `docs/results/scaled_rlvr_tier1b_result.md`. New code: `game_theory_llm/reasoning/reasoning_ops.py`, `tests/test_reasoning_ops.py`, `scripts/build_reasoning_ops_screen.py`.

### 2026-06-07: New-op headroom screen — closed-form ops saturate, search/bookkeeping ops have headroom (but token-walled) [result]

Screened the 3 new reasoning ops on the 30B base. Decisive pattern:

| op | type | base acc | verdict |
|---|---|---|---|
| **nim_grundy** | combinatorial (XOR) | 0.92–0.96 all depths | **ceiling** — 30B knows the Sprague-Grundy algorithm |
| **signal_abduce** | abductive (argmax+match) | **1.00** all depths | **ceiling** — trivial for 30B at any N |
| **opponent_id** | inductive (table-ID) | d1=0.58; d2+=0.00 @2560 → **d2=0.23, d3=0.17 @8192** | **viable** (token-walled) |

**Two through-lines:**
1. **Closed-form operations saturate for strong models.** Nim XOR and argmax-abduction have closed-form algorithms a 30B already knows → ceiling at every depth. Only **search/bookkeeping operations with no shortcut** retain headroom: deep game-trees (exponential leaves) and inductive table-ID (state tracking).
2. **Token walls are pervasive.** opponent_id d2 "floored" at 0.00/parse0.24 @2560 but is 0.23/parse0.95 @8192 — same wall as gametree d6 (0.00→0.21 @8192). Headroom operations are generation-heavy; **must train & eval with large token budgets**. Implication: Tier-1b's 2048-token TRAINING cap likely under-trained the gametree frontier.

**Viable training targets** = the 2 search/bookkeeping ops: **gametree** (deductive, d4–6) + **opponent_id** (inductive, d2–3), both needing ~4–8k tokens. nim/abduce would need much deeper/compositional instances to leave ceiling.

This refines the breadth experiment to a deductive+inductive 2-op curriculum trained WITH adequate tokens (tests breadth AND the training-token hypothesis together vs Tier-1b's 1-op/2048-tok run).

### 2026-06-09: Tier-2 breadth curriculum (deductive+inductive, 30B) — NULL; breadth dilutes in-domain, no transfer [result]

**Design**: GRPO on a 2-op tiered curriculum — gametree (deductive) d3→5 + opponent_id (inductive) d1→3, 200/cell = 1200 prompts, **6144 training tokens** (token-wall-safe). The only 2 of 5 candidate ops with 30B headroom (closed-form nim/abduce at ceiling). Run survived a Tinker billing outage (402 at b16, resumed from b10 checkpoint).

**Training = healthiest of the 3 runs**: at the hardest tier frac_mixed≈0.94, all_bad≤0.19 (vs Tier-1b's 31–44% all_bad at 2048 tok) — the bigger token budget kept GRPO gradient alive everywhere. Entropy rose tiers 1–2 then dropped at tier 3 (convergence). KL ~0.0022.

**Result**:
- **Transfer: mean +0.001** across 4 measurable held-out benchmarks (boolean +0.5pp, dyck −1.5pp, mmlu 0.0, bbh +1.3pp) — fails the ≥3-benchmark rule. Extrapolation flat (gt d6: 0; opp d4: +2.5pp noise). gsm8k control exactly 0. ✓
- **Breadth–depth tradeoff is proportional**: gametree d4 gain **+13.3pp with 200 prompts** vs Tier-1b's **+23.3pp with 500** — halving per-cell data halved in-domain learning, and bought zero transfer.
- **Inductive op is RLVR-resistant**: opponent_id ≤+3.3pp everywhere despite live signal — group-relative advantage on a small answer space rewards lucky guesses as much as genuine 2^k-table inference.

**Combined verdict (Tier-1 + 1b + 2)**: three pre-registered designs (scaled ops / deep single-op curriculum / breadth curriculum) all null on transfer at 30B. **Game-RLVR yields only local trained-cell skill; neither depth, token budget, nor op diversity converts it to general reasoning.** The 4B signals were small-model in-distribution effects. What held throughout: gsm8k no-regression (3/3 runs) and the methodology (headroom screens, token-wall checks, pre-registration) that made the nulls clean.

**If pushed further**: (a) data/epoch scaling at the frontier cell; (b) **process rewards** from the game solver (exact intermediate states are free — outcome-only GRPO may be the limiter); (c) breadth replication on a 4B/8B base where all ops have headroom.

Doc: `docs/results/scaled_rlvr_tier2_breadth_result.md`. Artifacts: `data/runs/gt_rlvr/{tier2_breadth_analysis.json, tier2_30b/grpo_curve.png, t2_base_*.json, t2_rlvr_*.json}`.

### 2026-06-10: Tier-3 process-vs-outcome reward head-to-head — process REJECTED [result]

**Design** (pre-registered): two GRPO arms identical except reward — OUTCOME (1{answer correct}) vs PROCESS (0.5·outcome + 0.5·order-aware LCS recall of the solver's gold internal-node values, spray-guarded). Same Tier-1b problem set (gt d3:200/d4:500/d5:500), 6144 training tokens, same hyperparams. Primary endpoint: d6 extrapolation, n=160.

**Primary (greedy, temp=0)**: base **0.144**, outcome **0.206** (+6.2pp, p=0.141), process **0.181** (+3.8pp, p=0.363; vs outcome **−2.5pp, p=0.572**). Process does NOT beat outcome — wrong direction in both decoding regimes.

**Secondary (temp 0.7)**: process *worse in-domain* (d5: 0.325 vs outcome 0.550 — splitting reward weight diluted answer pressure); transfer identically null both arms (+0.003 mean); gsm8k clean both. **Dead-group prediction mooted**: at 6144 training tokens both arms had all_bad=0.00 at the hardest tier — Tier-1b's 31–44% dead groups were token starvation, not a reward problem.

**Outcome-extrapolation thread**: temp-0.7 showed outcome d6 +15pp vs base (would be first positive extrapolation at 30B, implying training-token budget unlocks depth generalization) — but greedy shrinks it to **+6.2pp, p=0.141**. Consistent ordering (outcome > process > base in both regimes) but not significant at n=160. Needs n≥500/arm or multi-seed to confirm; firmly smaller than the +12.5pp in-domain gain.

**Methodological finding — eval stochasticity**: base d6 measured **0.212 / 0.113 / 0.144** across runs (temp-0.7 default in tinker_eval). Single temp-0.7 runs can manufacture/erase ±10pp on hard benchmarks. Small-effect endpoints must use greedy or multi-seed. (Prior tiers robust: conclusions rested on +23pp / Δ≈0 / multi-benchmark means.)

**Program status after 4 pre-registered designs** (scaled ops / deep curriculum / breadth / process rewards): game-RLVR on a strong 30B base reliably produces in-domain skill (≤+23pp, never regression) but **no validated transfer — across depth, operations, tasks, or reward signals**. Remaining untested lever: data/epoch scaling at the frontier cell (evidence to date: buys in-domain only).

Doc: `docs/results/process_vs_outcome_reward_result.md`. Artifacts: `data/runs/gt_rlvr/{tier3_analysis.json, t3_*, t3g_*_d6.json, tier3_{outcome,process}_30b/grpo_curve.png}`.

### 2026-06-10: Post-mortem — WHY game-RLVR transfer failed: slip-regime vs capacity-cliff [result]

Forensic analysis of existing per-item results (no new training). Three findings that jointly explain all four nulls:

**1. The model already knew the algorithm; RLVR taught +1pp execution reliability.** Compounding-slip model (acc = p^(2^d−1)) fits base d3–d5 almost exactly with per-node p=0.983 (ratios 1.10/1.00/0.93). The outcome arm's whole in-domain gain is p 0.983→0.993, which *predicts* d4=0.900 — exactly the observed 0.900. RLVR = slip-rate reduction, nothing more.

**2. d6 is a different failure regime, not more slips.** Slip model predicts base d6=0.343; actual 0.144 (ratio 0.42). With RLVR's reliability it predicts 0.642; actual 0.206. Past ~31 nodes the model loses the thread (state-tracking/retrieval over a growing trace) — a **capacity cliff**, which slip-rate training cannot cross.

**3. d6 competence is a lottery.** Three independent base runs on the same 80 d6 items: pass@1 0.212/0.075/0.150, pass@3 union 0.388, **solved-in-all-three = 0.000**. No item is stably known. RLVR arms' d6 solves are scattered vs the base's reachable set (outcome: 8/12 outside the union) — fresh lottery draws, not consolidated skill. (Also retro-explains the temp-0.7 d6 swings 0.113↔0.263.)

**Unified law: RLVR gains extrapolate within the slip regime and stop at the capacity cliff.** 4B "transfer" (+12.5pp) was same-regime reliability extension (its extrap depths sat at base≈0.34, on the slope); 30B asked the gain to cross a regime boundary. Cross-task null because execution reliability is task-embedded, not a general faculty. GRPO additionally has no usable gradient at the cliff (all-bad groups + lottery-noise variance).

**Answer to "it should have worked"**: the intuition assumes the bottleneck is knowing the method (true for human learners). The 30B never lacked the method (d3=0.975, one reliability parameter explains d3–d5); it lacks long-horizon state tracking — a capacity property that policy-gradient distribution-shaping doesn't expand.

**If ever resumed, target the cliff, not the policy**: decomposition/structured-scratchpad training (chunk + cache subtree values), tool-use external memory (cliff = retrieval precision), and a cheap dissociation probe (b3-d3 13 nodes vs b2-d6 63 nodes) to separate recursion depth from bookkeeping load.

Doc: `docs/results/rlvr_transfer_postmortem.md`.

### 2026-06-12: Memory-harness ladder steps 1+2 — cliff mechanism CONFIRMED; zero-shot memory use fails; strategy is the missing piece [result]

All on OpenRouter qwen3-30b-a3b-instruct-2507, temp 0, within-provider comparisons (Tinker key revoked server-side mid-program — confirmed via raw curl; steps 1+2 are sampling-only so ported to `harness_or.py`).

| condition | d6 | d7 |
|---|---|---|
| plain (single call, 8k) | 0.163 | 0.087 |
| **memory harness (zero-shot)** | **0.006** | **0.013** |
| **enforced decomposition** | **0.988** | — |

**Step 2 = decisive mechanism confirmation**: env drives post-order traversal, model answers only local min/max (own values propagated) → d6 **0.16→0.99**, local reliability **0.9964** over ~5,000 sequential local steps. The capacity cliff is entirely state/control; local computation is intact.

**Step 1 = zero-shot memory use FAILS** (worse than no harness). Forensics: the model uses memory as a **transcription buffer, not a computation cache** — fills notes with the 64 *leaf values* already in the prompt (mean 90 notes), then crams the whole 63-node computation into the final round anyway, making the same lost-place parity errors (e.g. `B.B.B = max(−6,−2)` at a MIN node). My pre-stated prediction (60–75% harness lifts d6) was WRONG; the slip-regime-restoration prediction was RIGHT.

**Perfect setup for Step 3**: gap exactly localized — local skill 0.996, env-managed strategy 0.988, model-managed strategy 0.006. The only missing ingredient is the note-taking/decomposition *policy* — a behavioral object RLVR can actually train (unlike capacity, which Tiers 1–3 proved it can't). Headroom 0.006→~0.99; signal exists (harness allows round-1 direct answering at d4–d5 → mixed groups).

**Step 3 BLOCKED on Tinker credential** (the unchanged key that ran Tiers 1–3 now 401s at the auth endpoint — server-side revocation, likely from the billing episode). Env/launcher/curriculum built & committed (`gt_memory_env.py`, `tinker_grpo_mem.py`, `train_tier4_mem.jsonl` d4:150/d5:300/d6:350); fires on next successful auth probe.

Doc: `docs/results/memory_harness_result.md`.

### 2026-06-16: Guided-strategy probe (step-3 ceiling) — knowing the algorithm does NOT help; step-3 run blocked on billing again [result]

**Guided probe** (OpenRouter qwen3-30b-a3b-instruct, temp 0): handed the model the exact bottom-up caching algorithm (path notation, MAX-even/MIN-odd parity, "cache nodes whose children are known"). Result: **d6=0.025, d7=0.062 — BELOW plain (0.163/0.087).**

Full ±state picture now:
| condition | d6 |
|---|---|
| plain | 0.163 |
| zero-shot memory | 0.006 |
| **guided (explicit strategy)** | **0.025** |
| enforced decomposition (env-managed) | **0.988** |

The explicit strategy **fixes the parity errors** (zero-shot's signature failure) and produces structurally-correct caches, yet scores below plain because the model dumps all 63 node computations in ONE round (mean 1.1 rounds), maximizing slip surface. **Knowing the procedure is not the bottleneck; executing 63 bookkeeping steps without losing a value is.** Every model-manages-state condition fails (0.006–0.16); only env-manages-state works (0.988). → **low ceiling estimate for step-3 RLVR.**

**One escape hatch guided lacked**: step-3's RL env caps each round at 1600 tokens — physically too short to cram a d6 tree — forcing work across rounds with state in notes (the decomp-like regime that scores 0.99). Step-3's real job is reframed: *induce chunking*, not *teach the rule*. Whether GRPO can beat the model's cramming prior is the open question.

**Step-3 status**: new key `tml-eZK22U…` authenticated; multi-round memory env smoke-tested OK (2.6 rounds, 11.6 notes, reward from correctness). Full run launched but **died pre-first-checkpoint when the account re-entered a 402 billing block** (same balance-exhaustion as Tier-2). Env/launcher/curriculum committed; re-fires on billing restore.

Doc: `docs/results/memory_harness_result.md`.

### 2026-06-16: Forced-chunk probe completes the surrogate — ALL model-managed conditions fail; only env-managed works [result]

Final no-training datapoint for the step-3 question. Forced chunking via prompt (≤600 tok/round, ≤18 rounds, strict "compute ≤4 ready nodes, save NOTEs, CONTINUE; answer only when A & B cached"): **d6 = 0.000 (n=48)** — the model NEVER produced an answer in 18 rounds, caching only 7.5 of 63 nodes. Forced to chunk, it can't run the protocol (loses track of ready nodes, stalls).

Complete ±state ladder on d6:
| condition | d6 | manager |
|---|---|---|
| plain | 0.163 | model |
| zero-shot memory | 0.006 | model |
| guided (explicit algorithm) | 0.025 | model |
| **forced-chunk (strict protocol)** | **0.000** | model |
| **enforced decomposition** | **0.988** | environment |

**Conclusion**: the deficit is not strategy knowledge, not cramming, not parity — it is the inability to *reliably execute a long self-directed bookkeeping loop*. A capability property of in-context multi-step control; prompting cannot install it. → step-3 RLVR ceiling is LOW (GRPO has near-zero positive signal at d6 — correct answers are lottery draws, forced protocol never completes — and Tiers 1–3 showed RLVR moves execution reliability ~1pp). Honest prediction: small/no uplift. The RL run is still worth running to confirm (the env's 1600-tok/round cap creates a chunking path the prompted probes lacked), but expectations low.

**Step-3 (RLVR training) remains BLOCKED on Tinker billing (402)** — external, user-action-required; new key authenticates but balance exhausted. Run staged, auto-fires on billing restore.

Doc: `docs/results/memory_harness_result.md`.

### 2026-06-24: Step-3 memory-RLVR — RL INDUCES the chunking behavior prompting couldn't; accuracy payoff pending eval [result]

The multi-round memory GRPO run (30B, gametree d4/d5/d6 curriculum, 1600 tok/round, ≤6 rounds) survived repeated Tinker outages (402 billing + transient 404 'Promise not found' on long late-curriculum episodes) via resume-from-checkpoint, reaching **batch 20/34 (62%)** before a deterministic batch-22 failure (d6 episodes grow to 5+ rounds × ~63 notes → server context/promise limit) + renewed billing block. Checkpoint: `…2e381db3…/sampler_weights/000020`.

**Behavioral finding (decisive, training phase)**: notes **15.7→63.0**, rounds **3.8→5.4**, jumping exactly at the d6 tier — RL drives the model to write ≈the full 63-node tree across 5+ rounds (cache-bottom-up across rounds). This is the policy that zero-shot memory (90 notes, all leaves, 1 pass), guided (1.1 rounds), and forced-chunk (stalled 7.5 notes) all FAILED to produce. **RLVR moved the policy where prompting could not** — confirms step-3's premise that note-taking strategy is a trainable policy-level object (vs Tiers 1-3 where RLVR could only nudge slip rates ~1pp).

**Open question**: does induced chunking lift ACCURACY? In-training d6 correct stays low (0.026), but that's the curriculum's hard tier under temperature, not the clean test. Held-out greedy d6/d7 trained-vs-same-provider-base-in-harness needs the Tinker eval suite (402-blocked again). Early hint sobering: low in-train d6 acc suggests per-node slips still compound over 63 self-directed steps even with chunking induced (consistent with low-ceiling prediction). Eval armed to auto-run on billing restore.

Infra note: long-episode RL (5+ rounds, 63 notes) deterministically hits a Tinker promise/context limit at batch ~22 — would need shorter rounds or fewer max-rounds to train d6 to completion. Doc: `docs/results/memory_harness_result.md`.

### 2026-06-24: Step-3 memory-RLVR FINAL — behavior induced, accuracy NOT; low ceiling confirmed [result] [COMPLETES /goal]

Held-out greedy eval of the batch-20 memory-RLVR checkpoint. **Verdict: RL induced the chunking behavior but did not lift accuracy — only environment-managed control crosses the d6 cliff.**

| condition | d6 | d7 |
|---|---|---|
| **trained, in-harness** | **0.062** | **0.000** (parse 0.21) |
| trained, no-harness (free 8k) | **0.175** | — |
| base, no-harness (Tinker greedy) | 0.144 | — |
| base, in-harness (OpenRouter, cross-provider) | 0.006 | 0.013 |
| env-managed decomposition | **0.988** | — |

**Three within-provider facts:**
1. **Harness hurts even the trained model**: trained in-harness 0.062 < trained no-harness 0.175. After RL trained the memory strategy, the model still solves d6 better by reasoning freely than by operating its trained protocol.
2. **RLVR didn't move free reasoning**: trained no-harness 0.175 ≈ base no-harness 0.144 (noise) — same null as Tiers 1–3.
3. **Behavior induced, accuracy not**: RL drove notes 16→63 (train)→77 (d6 eval)→163 (d7); model genuinely caches the tree across rounds (prompting couldn't). But d6=0.062, d7=0.000 — writing notes ≠ getting 63–127 node computations right; per-node slips compound over the self-directed loop exactly as the post-mortem predicted.

**Capstone across the whole arc**: the d6 backward-induction cliff is a multi-step *control/execution* deficit, not knowledge. Environment-managed decomposition crosses it (0.988); NOTHING that leaves the base policy to self-manage state does — not free reasoning (0.16), not zero-shot memory (0.006), not guided strategy (0.025), not forced chunking (0.000), and not RL-trained memory strategy (0.062). The path to crossing the cliff is architectural (tool/env manages state), not training the base policy. This mirrors the Tiers 1–3 finding (game-RLVR gives in-domain slip-rate gains, no transferable reasoning) at the mechanism level.

Infra caveat: training capped at batch 20/34 (long d6 episodes hit a Tinker server promise/context limit at batch ~22); recurring 402 billing exhaustion throughout. Doc: `docs/results/memory_harness_result.md`. **All 3 ladder steps now run + reported.**

### 2026-07-01: Why synthetic game-tree RLVR didn't transfer to long-form reasoning — and the untried fix [brainstorm]

**Question**: post-mortem on the whole RLVR arc (Tiers 1–3 + memory-harness ladder): why did synthetic game-tree data produce in-domain gains (+23pp) but zero long-form/transfer improvement?

**Diagnosis (each claim tied to a measured result):**
- **Trained the wrong variable.** Long-horizon acc ≈ p_local^N. RLVR moved local reliability ~1pp (0.983→~0.99) — compounds in-domain, but p_local was already near ceiling (0.996 env-managed). The barrier at depth is **control state** (whose turn / which nodes done / what next), not local computation: env-managed decomposition = **0.988** at d6 vs ALL model-managed conditions 0.006–0.16.
- **Outcome-RL can't reinforce what the model can't sample.** At d6 correct completions are lottery draws (pass@3 union 0.388, solved-in-all-3 = 0.000) → GRPO groups all-fail → zero advantage → no gradient. Process reward (Tier-3) didn't help because the failure is the orchestration loop, not intermediate values.
- **The task format hid the skill.** Shallow trees fit one window → training teaches "cram in one pass", exactly the strategy that fails at depth (guided probe: handed the algorithm, model still crammed 1.1 rounds, 0.025 < plain 0.163). The needed skill — a self-directed bookkeeping loop — was never elicited by the training distribution.
- **Single-op narrowness**: minimax notation shares little surface with prose reasoning; ops that might have (Nim, abduction) saturated.

**Proposed fixes (ranked):**
1. **SFT on synthetic bookkeeping traces (untried, most direct).** We can algorithmically generate perfect multi-round demonstrations (post-order traversal, explicit NOTE writes, ready-node selection, resume-from-notes). Behavior-clone at d4–d6, then RL to harden. Directly fixes the reachable-set failure.
2. **Dense state-level reward in the multi-round env**: reward correct notes per round (ground truth known per node) → signal at d6 even when episodes never finish. Distinct from rejected Tier-3 process reward (that rewarded values inside a single cram).
3. **Curriculum pinned at the cram boundary** (d5→d6) so GRPO groups are mixed and chunking earns differential advantage.
4. **Diversify long-horizon surface forms, hold the skill constant** (arithmetic ledgers, frontier-list graph search, register machines): make explicit state maintenance the only shared bottleneck (≥3-benchmark rule applied to training).
5. **Honest alternative — scaffold, don't train**: env-managed control = 0.988 today with zero training; if the use case tolerates a harness, that IS the fix.

**Recommendation**: fixes 1+2 together on the existing `gt_memory_env.py` infra (env/launcher/curriculum exist; trace generator is a small addition). This is the experiment the probe ladder points to but was never run (blocked on billing at the time).


### 2026-07-02: Tier-5 design locked — domain-general inline ledger, SFT→RL [design]

**Decision**: attack the bookkeeping-loop deficit with an **inline state-ledger protocol** (portable, no harness at inference), trained SFT→RL. Spec: `docs/superpowers/specs/2026-07-02-tier5-inline-ledger-design.md` (worktree).

- **Protocol invariant**: LEDGER / NEXT / NOTE / CHECKPOINT / ANSWER — byte-identical grammar across all families; only content varies. Checkpoint re-prints = inline analogue of env-managed state refresh (the 0.988 regime).
- **Families**: train on 4 (minimax trees, register machines, graph search, forward chaining); **hold out 2 entirely** (object tracking, scheduling) + ≥3 real benchmarks (BBH multistep-arith, tracking-shuffled-objects, dyck; GSM8K short-form control).
- **Recipe**: SFT on generated gold traces incl. **~10-15% recovery traces** (corruption→verify→correct; attacks exposure bias) → eval gate → GRPO with **verifier-based dense ledger reward** (answer + λ·per-NOTE accuracy; spray-guarded precision·recall). Verifier built once, used for both tests and reward.
- **Attribution**: unprompted-usage rate + suppression ablation, so uplift is credited to the technique, not generic FT.
- **Tinker billing confirmed LIVE** (2026-07-02 probe: auth + training-client OK) — the June blocker is gone.
- **Honest-null clause**: if SFT can't install the behavior even in-domain, the capability-property conclusion strengthens; stop and report.


### 2026-07-16: 5-game steering study COMPLETE — coop vector replicates in 4/5 games [result]

**The cooperation vector (fit on one-shot PD vignettes, Gemma E4B L16) causally steers live multi-agent play across game genres.** Full report: `docs/results/steering_in_games_report.md`.

| game | genre | result |
|---|---|---|
| One Night Werewolf | social deduction | bidirectional dose-response (aggr −16.2 @+4, p=0.0004) |
| Secret Hitler | social deduction | replicated; Liberal win 56→68→84% monotone |
| **Risk-lite** | territorial conquest | monotone −4→0→+2 all 3 indices; −4 all sig (trust p=0.0009) |
| **Monopoly-lite** (new engine) | economic negotiation | **strongest contrast in study: aggr +19.2 @−4, p=4.2e-06**; coop −13.0 p=0.004 |
| **Diplomacy-lite** | full-press negotiation | **underpowered null** — 30–41% of actions are parse-fallback noise (competence floor) |

- **Anti-cooperative direction is the universally robust channel** (significant in all 4 measurable games); pro-social side ceiling-limited where baselines are already cooperative (SH/Risk/Monopoly pattern).
- **Methods findings:** (1) usable α is game-dependent — +4 collapses Risk's fragile format into template-stub muteness (57% stub msgs) while ONW/SH tolerate it; (2) always check the fallback/parse floor before interpreting judge indices (Diplomacy fails the floor at ~35% noise); (3) framework gap found: alliance-accept affordances were never rendered into prompts → formal pact metrics dead across ALL studies incl. the original two.
- Monopoly engine built from scratch for this study (trading + per-turn negotiation as cooperation surface; v1 sweep discarded after dormant-negotiation discovery, v2 = the result).


### 2026-07-17: Tier-5 inline-ledger FINAL — in-family skill, anti-generalization off-family, RL null [result]

**Verdict (pre-registered §7 discipline): generalization claim REJECTED.** Full doc: `docs/results/tier5_ledger_result.md`.

- **In-family the technique is real and load-bearing**: extrapolates +45–55pp beyond trained horizons (graph h130 .45→.90, chain h130 .45→1.00); suppressing the ledger collapses in-family performance (.90→.05).
- **Off-family it is actively harmful (anti-generalization)**: held-out tracking .78→.08 (−70pp), BBH-arith .976→.32. Causal decomposition via suppression ablation: BBH-arith harm = **format imposition** (recovers to .948 with one instruction); tracking harm = **family-specific capability damage** (no recovery). Double dissociation clean.
- **Style diversity ≠ skill generality**: 5 naturalization styles didn't buy family transfer — family coverage was the binding constraint (the SAE lexical-artifact lesson at the behavioral level). The when-to-deploy discrimination only protected what the mix explicitly covered (GSM8K unharmed at .92).
- **The d6 cliff survives everything**: SFT (format-perfect, values corrupt: h63 acc .017 @ parse 1.0) and RL. **RL = ±0.02 everywhere** — third independent confirmation that outcome-RL moves execution ~1pp. Register n150 = token-overhead trap (ledger cost scales with horizon; base solves tersely .63, SFT .00).
- **Capability-property conclusion strengthened**: trained bookkeeping *format* ≠ transferable bookkeeping *competence*; env-managed control (0.99) remains the only regime that solves depth. Next-if-ever: train the decision-to-externalize across many families, or scaffold at inference.


### 2026-07-22: Mechanics audit, clean-engine ONW replication, tier-5 v2 null, AAAI-27 package [result]

**Game-mechanics audit** (all 5 play engines + shared infra; 6 parallel auditors + 6 adversarial verifiers): **42 findings confirmed, 0 refuted**. Paper-affecting: ONW night-action broadcast leak (roles outed into every seat's prompt), SH sequential open ballot (documented as simultaneous/secret), whisper double-count doubling private lines in the judge transcript (SH public-msg ratio distorted ~46% rel.). Report: `.worktrees/steering/docs/results/game_mechanics_audit.md`. Steering-report-affecting: Monopoly α=−4 headline effect sits on **58.8% parse-error** (report claimed ~7%, baseline-only); Diplomacy true order-turn fallback **94.6%** with SUPPORT structurally impossible under fallback ("no directional bias" withdrawn); Monopoly trade metrics were artifacts (100% of trades came from random fallback). Fallback *uniformity* itself verified correct in the shared runner. Corrections folded into `steering_in_games_report.md`.

**ONW v2 re-run** (fixed engine, 125 matches, matched seeds): **the steering X-cross replicates, stronger than v1** — coop α=−4: coop 42.4 / aggr 60.5 (p=.0013/.0003); baseline 54.8/40.7; coop α=+4: 63.6/23.3 (p=.014/.0009). Trust vector stays null (good negative control). **Win-rate story weakened**: baseline village win 0.32→0.48 on the clean engine; +8pp at α=+4 now underpowered at n=25 — paper phrasing to be softened. SH v2 in flight.

**Tier-5 GRPO v2** (eval-disjoint pool, seeds 300000+, 0 overlap): mean RL−SFT delta **+0.021** vs v1's −0.017 — **the RLVR null is confirmed clean of the train-on-test contamination**. Only mover: BBH arith +0.156 (partial format recovery, still −0.50 below base). Generalization-claim rejection unchanged. Commit `44ce379`.

**AAAI-27 package built in one day** (abstract deadline was 7/21 AoE; full paper due 7/28): `aaai27.tex` exactly 7 content + 2 ref pages (claims inventory: **zero content drift** from the NeurIPS draft), 7-page supplementary, 31-item repro checklist, anonymized code zip w/ MIT license. Remaining: SH v2 → gameplay-number swap → final QA.

### 2026-07-22: Cooperation-measurement audit — judge indices measure talk, not play [result]

Focused follow-up to the 2026-07-21 mechanics audit: reproduced the exact judge pipeline (`build_transcript()[:9000]` from `scripts/analyze_game_steering.py`, post-dedup-fix) over ALL production logs for the five games. Full report: `.worktrees/steering/docs/results/cooperation_measurement_audit.md` (commit 272836d).

- **Construct validity (all 5 games)**: the judge transcript renders only messages, alliance events, `vote` actions, and the outcome line — **zero economic/military/order actions** in any game. Cooperation/trust/aggression indices are ratings of table-talk.
- **Truncation**: `transcript[:9000]` head-truncates — the judge saw the OUTCOME in **1/125 SH matches** (median transcript 22.7k chars), 23/100 Monopoly, 105/125 ONW. The paper's SH judge indices rate the first ~40% of talk, no outcome.
- **SH vote schema bug**: `build_transcript` reads `action["target"]` (ONW schema); SH votes are `{"ja": bool}` → all 6,709 SH vote lines render as `P<i> VOTES PNone`. Judge never sees a vote direction, nomination, or policy enactment.
- **Monopoly trades**: full-run count — **145/145 `propose_trade` actions fallback-originated** (126 in the α=−4 parse-collapse condition); models are told trades are "listed in your actions" but they're never rendered and no grammar can select one.
- **Monopoly honour is definitional**: `judge_alliance` no-op + "unbetrayed ⇒ honored" bucketing → 201/201 accepted alliances honored, betrayal_rate ≡ 0 structurally.
- **Unaffected**: win rates / `coop_side_win` (read terminal record), message metrics post-dedup. ONW's judge path is the healthiest (correct votes + roles context).
- **Decision pending**: fixes (schema-aware transcript, head+tail windowing, trade grammar) all change the judge's input → indices not comparable across the fix; SH v2 re-analysis inherits these issues unless fixed first.

### 2026-07-22: SH v2 complete — obs-layer leaks proven decision-inert; AAAI package final [result]

**SH v2** (125/125 via Modal call-ID salvage, 0 respawns): −4 arm significant (coop Δ−10.4 p=0.0015, aggr Δ+13.4 p=0.0025), +4 saturated as in v1, **Liberal win 0.56→0.68→0.84 monotone (identical to v1)**. Full-range −4 vs +4: p≈4×10⁻⁴ both indices.

**Provenance discovery**: SH v2 replays v1 **byte-identically** (temp 0.7, fully seeded) despite the demonstrably fixed engine → the production `_LocalSteeredPlayer` (in `modal_app.py`) drops `action`-type obs, so **neither the ONW night-action leak nor the SH ballot leak ever reached a decision prompt**. The whisper double-count (judge transcripts) was the only real contamination channel of published numbers; v2 analysis corrects it. ONW v1→v2 deltas = engine evolution since May, not leak fixes. Audit addendum: `.worktrees/steering/docs/results/game_mechanics_audit.md`.

**Also caught by verification**: `n_fallback` is a raw count, not a rate — true SH fallback 6.2%/8.6% (baseline/α=−4), ONW ≤1.3%; paper corrected.

**AAAI-27 package FINAL**: `aaai27.tex` 7+2pp with v2 gameplay numbers (every value + p independently recomputed from per-match data), regenerated figure, supplementary 7pp, checklist, code zip. All QA gates green. Remaining: OpenReview uploads (paper by 7/28, supp+code+checklist by 7/31).

### 2026-07-23: Games-platform rebuild — parse-void, plain Monopoly, promise ledger [result]

Executed the 7-task SDD plan fixing the cooperation-measurement audit findings (spec + plan in `.worktrees/steering/docs/superpowers/{specs,plans}/2026-07-22-plain-monopoly-parse-policy-promise-judge*`). Branch `feature/activation-steering`, range 7ebf476..bdcae3a, 16 commits, all task reviews + final whole-branch review clean.

- **Diplomacy deleted** (engine/map/tests/all references) — 2,912 lines removed.
- **Parse-void policy** (all games): no more random-move fallback; 1 initial + 3 informed retries (model sees its parse error), then the match VOIDS with an `aborted` record naming model/steering/phase. `cancel_rate` is now a headline per-condition metric — the "steered too far" signal.
- **Plain Monopoly rebuild**: auto-rolled dice, strict tag grammars (`<decision>`, `<trade>`, `<response>` — negation-safe, conflict-detecting), full public state in every prompt (all positions + full ownership table; others' cash hidden), one end-of-turn trade dialogue with free-text messages. 104 engine tests.
- **Analyzer**: game-aware FULL transcripts (SH ja/nein votes + nominations + enacted policies; Risk attacks; Mono events + trade dialogue; ONW byte-compatible), 60k head+tail judge window (outcome always visible), trade metrics, coop_side_win=None for sideless games.
- **Promise ledger (all 4 games)**: second Sonnet judge pass extracts every commitment → kept/broken/unresolved with quoted evidence; `renege_rate` replaces the mechanical honored-by-default alliance buckets. Raw ledgers stored in per_match.json.
- Suite: 641 passed (+2 known network-dependent failures). Legacy run dirs still analyze cleanly.
- **Modal spend tracking** added (`scripts/modal_spend.py`): ~$1,640 cumulative across all game-steering runs; discovered sh_steering_v2 is actually COMPLETE (125/125 with terminals).
- **Next**: any new steering run needs `modal deploy` first (workers bake engine code); SH v2 can be re-analyzed with the fixed pipeline; fresh Monopoly runs would exercise the real trade dialogue for the first time.

### 2026-07-24: NeurIPS 2026 reviews in — scores 2/5/2 [reviewer]

Meta (AC Nuwt) weaknesses: (1) organization/writing, (2) add popular open-weight LLMs (Gemma) to the behavioral eval, (3) missing activation probing/steering related work.

**Recurring across reviewers (bSST=2, 9URN=5, Z6xa=2):**
- **Behavioral↔mechanistic disconnect** (all three + meta): Gemma is the probing/steering model but absent from the Section-3 behavioral battery. Most actionable single fix.
- **Missing steering citations**: ActAdd (2308.10248), RepE (2310.01405), CAA (2312.06681); plus Akata 2023 (2309.05898), Lorè&Heydari (2305.16867).
- **Reproducibility** (both reject reviewers): no anonymous code/data at review time; prompts not shown. (AAAI package already fixes: anonymized code zip + supplementary.)
- **Presentation**: Table 1 ≡ Fig 1 (drop one); Fig 3+4 combinable; Fig 2 too small; "wee"→"we" L31; missing citations L111-113 (already repaired in aaai27.tex); undefined symbols (level 0/1 L133, RSA L207).
- **Methodology**: Opus generates + validates + is evaluated → self-preference concern; no human vignette audit; "negligible framing effect" needs stronger manipulations (9URN, who still voted accept).

**Decision (user-approved)**: AAAI-27 version gets the cheap fixes AND a Gemma-family behavioral battery (OpenRouter, API-only) before the Jul 28 upload; human audit / strong-framing conditions / generator-circularity banked for the next cycle.

### 2026-07-24: Gemma battery closes the NeurIPS reviewers' top criticism [result]

Ran the Section-3 behavioral battery on the Gemma 4 family (the paper's steering models) and folded it into the AAAI-27 draft (paper commits 461f98e + 616acff).

- **26B-A4B + 31B**: all 3,010 vignettes via OpenRouter with the frontier harness (~$1). **E4B-it**: not hosted anywhere, so evaluated on our Modal workers at steering α=0 — full battery would've cost ~$57 (27.5s/story unbatched), so stratified 10/cell subsample (~$13; 12% cap-truncation disclosed, concentrated in Chicken/BoS).
- **Result: the steering models are behaviorally ordinary members of the evaluated population.** Game-ordering Spearman ρ vs frontier mean = 0.94 / 0.83 / **1.00 (E4B)**; overall A-rates 0.60–0.65 inside the frontier envelope (0.61–0.69); **0/15 framing contrasts survive Bonferroni** — payoff-dominance + negligible-framing extend to open-weight models at every family scale.
- Paper: new bridge paragraph + table before Mechanistic Analysis; limitations updated; ActAdd/RepE/CAA related-work added; Table 1 dropped (dup of Fig 1); level-0/1 coding defined; generation+judge prompts verbatim in supplementary; XGBoost table → supp to hold the 7-page budget. QA green: 7 content + 2 ref pages, fonts embedded, 0 undefined refs, anonymity clean.
- Scripts committed in main repo: scripts/run_gemma_evals.py, scripts/analyze_gemma_evals.py, gemma family in MODEL_REGISTRY.

### 2026-07-26: External-benchmark "transfer" is a one-sided degradation — Table 3 was missing its baseline [result]

Audit triggered by a single question during an AAAI-27 writing pass: *was there actually any transfer to the external moral-reasoning benchmarks?* Recomputed every cell from the raw Modal shards (`local_data/pd_full_v1_shards_*`, `pd_26B_A4B_v1_shards_*`). **Answer: yes, but only in the anti-cooperative direction, and it is a degradation, not an attitudinal shift.**

**The reporting gap was an omitted column.** Table 3's strong-steering block showed only $\alpha_-$ and $\alpha_+$. But `run_powered_evals.py` swept `ALPHAS = (-6, -3, 0, +3, +6)` — the $\alpha=0$ baseline was collected all along and simply never displayed. Adding it inverts the reading.

**MoralChoice high-ambiguity** (the headline, $p=5\times10^{-4}$ / $2\times10^{-3}$):

| α | 26B-A4B | E4B (n=300) |
|---|---|---|
| −6 | 0.540 | 0.507 |
| −3 | 0.630 | 0.582 |
| **0** | **0.750** | **0.636** |
| +3 | 0.750 | 0.612 |
| +6 | 0.778 | 0.635 |

- **The positive arm does nothing on either scale.** 26B: +0.028 vs baseline, $p=0.74$. E4B: **−0.002**, $p=1.0$ — zero to three decimals. On 26B the whole positive limb wanders 0.770 / 0.750 / 0.740 / 0.778 around a 0.750 baseline: noise.
- **The entire Δ is the negative arm collapsing toward chance**: −21.0 pp ($p=0.003$) on 26B, −13.0 pp ($p=1.6\times10^{-3}$) on E4B, against 0.50 chance on a binary task. Cleanly monotone: 0.750 → 0.720 → 0.630 → 0.600 → 0.540.

**ETHICS-Utilitarianism** shows the *mirror* asymmetry: E4B $\alpha=-6$ is exactly at baseline (0.870 vs 0.870, $p=1.0$) and only $\alpha=+6$ declines (−6.3 pp, $p=0.046$); 26B does not replicate ($p=0.83$). **No benchmark in the table rises above its unsteered baseline at any α.**

**Two further problems found:**
- **At α=±3 all eight contrasts are null** ($p$ = 0.092 … 1.0). Nothing transfers at the standard dose; the effects appear only after doubling α *and* tripling n.
- **No multiplicity correction.** 12 contrasts, uncorrected $p$. Bonferroni threshold is $p<0.0042$: the two MoralChoice rows clear it, the ETHICS-Util row ($p=0.046$) does not — yet it sat in the abstract as "a small but significant welfare-comparison degradation." Inconsistent with the paper's own Bonferroni discipline on the 35 behavioral and 15 Gemma contrasts.

**What survives, and it is worth having.** Anti-cooperative steering selectively drives MoralChoice high-ambiguity to chance on **both** model scales while ETHICS-Util, ETHICS-Deontology, MMLU, GSM8k and HumanEval stay flat, at ~99% parse rates. The capability nulls are exactly what defends this against "the model just broke" — damage is confined to the ambiguous moral dilemmas. This is a **selective-disruption** result, not moral-attitude transfer.

**Paper updated** (`aaai27.tex`): α₀ column added to Table 3; §Transfer rewritten around the one-sidedness; Bonferroni threshold stated in text and caption; abstract / intro / discussion / conclusion / limitations rescoped to "the direction *removes* cooperative behavior off-distribution, it does not install it." Held to 7+2 pages. Figure 5 converted `figure*`→`figure` — it was a 0.38-width image inside a full-width float, so the change bought back a column *and* fixed the NeurIPS reviewers' "figure too small" complaint.

**Generalizable lesson:** a two-arm Δ with no baseline column cannot distinguish a bidirectional shift from a one-sided collapse. Every steering table should carry α=0. Worth re-reading the gameplay tables the same way — they do report α=0, but the win-rate rows have not been checked baseline-relative.

### 2026-07-26: Setpoint moved, sensitivity didn't — temporal analysis + two more normalisation bugs [result]

Follow-on from the same day's transfer audit. Same failure mode recurring: **a difference that looks larger than it is until the right normalisation is applied.** Three instances found and fixed in `aaai27`.

**1. Probe comparison was class-count confounded.** Game probe is 7-way (chance 0.143), context probe 10-way (chance 0.10). Raw accuracies aren't comparable, and neither is ratio-to-chance — it's bounded by $1/\text{chance}$, so the two probes had *different ceilings* (7× vs 10×). Ratio therefore favours the probe with more classes, and on that metric **the context probe wins in-distribution (7.00× vs 6.51×)** — the old "despite having more classes" phrasing was one division away from being flipped by a reviewer. Now chance-corrected, $(acc-chance)/(1-chance)$: **0.92 vs 0.67** in-distribution, **0.70 vs 0.44** cross-transfer. Claim holds comfortably.

**2. RSA promoted to primary evidence, with its own correction.** Same-label base rates differ (14.2% of pairs share a game vs 9.9% a framing), inflating $r_\text{game}$ by 1.17× on a point-biserial argument ($r = d\sqrt{p(1-p)}$). **5.60× → 4.79×** — barely moves. RSA is the class-count-robust measure, so it should have been primary all along; the "24-pp gap" no longer carries that label.

**3. New: does framing sensitivity change across model generations?** 27 dated checkpoints, 5 families, May 2024–Jun 2026, 13,315 parsed decisions (`scripts/analyze_framing_over_time.py`).

- **0 of 135 (model, dimension) contrasts survive Bonferroni.** Uncorrected 15/135 vs 6.8 expected. The negligible-framing result now extends back two years and across 20 more models — a real strengthening of the headline claim.
- Raw framing effect *does* decline with release date (ρ = −0.445, p = 0.020) — **but this is a ceiling artifact.** At A-rate 0.996 the largest achievable |Δ| is 0.008, so near-ceiling models *cannot* show a framing effect. Cooperation rate predicts effect size better than date does (ρ = −0.540, p = 0.004).
- Permutation null preserving each model's own marginal A-rate and group sizes → ρ = −0.363, p = 0.063. Headroom-normalised → **flat, ρ = −0.057, p = 0.78.**

**4. And the cooperation trend itself was overstated.** The supplementary asserted "near-universal convergence toward high cooperation across all families" with no statistic. Tested three ways, and they disagree — which is the point:

| test | result |
|---|---|
| model-level monotone (Spearman, n=27) | ρ=+0.20, p=0.31 — null |
| model-level median split (13v13, 0.818→0.911) | p=0.027 — but post-hoc, ignores clustering |
| family-level sign test (n=5) | 4/5 up, p=0.375 — null |

GPT trends **down** (0.919→0.832), so "all families" was wrong. Spread narrowing is not significant either (SD 0.130→0.080, Levene p=0.43), so "convergence" is gone. Most of the movement is Claude's discontinuity at Opus 4.5 — one vendor's alignment update, not an industry trend.

**Synthesis (now in the supplementary):** what drifted is the cooperative **setpoint**; what did not move is **sensitivity** to narrative framing. The two are mechanically coupled — a rising setpoint compresses the headroom framing can act in, so constant sensitivity registers as a shrinking effect size. That's exactly why holding each model's marginal fixed under permutation dissolves the apparent decline. Alignment training moved *where* models sit, not *how much context pushes them around* — the same setpoint/sensitivity split the steering work exploits by moving the setpoint at inference time.

**Generalizable lesson (third time this session):** before believing any cross-condition difference, ask what its maximum possible value is under each condition. Transfer table → missing α=0 baseline. Probe comparison → different ratio ceilings. Framing-over-time → different headroom. All three looked like findings and were partly or wholly artifacts of an unequal denominator. See [[project_external_transfer_one_sided]].

Paper commits `2c5f75a` (+ `14521f3`, `27fb2e0` earlier same day). Also fixed a 5th instance of the one-column/two-column figure-overflow bug (layer-probe subfigures).

### 2026-07-27: Mixed-population steering in ONW — no contagion; and steering moves talk, not play [result]

Two connected findings. Both say the same thing from different directions: **the steering vector changes what agents say much more than what they do.**

**1. The published ONW α=+4 effect is largely an artifact of what the judge could see.**

Whole-table judge, same 25 matches, same prompt — only the transcript builder swapped:

| condition | OLD (talk-only, `[:9000]`) | NEW (all actions, full) | paper |
|---|---|---|---|
| coop α=−4 | 42.0 | 43.5 | 42.4 |
| baseline | 54.8 | 54.7 | 54.8 |
| coop α=+4 | **63.7** | **57.4** | 63.6 |

The old builder reproduces the paper to within 0.4pp on all three cells — strong validation the pipeline is right. Per-match tests: **α=+4 vs baseline is +8.0 (p=0.025) on talk-only but +3.4 (p=0.249) once actions are rendered.** α=−4 survives both (−13.2 p=0.001 → −11.2 p=0.0035). *An earlier truncation-only test came back clean and I wrongly reported it as clearing the paper — it held action-rendering fixed in both arms, so it was blind to the variable that mattered.*

**2. Mixed-population battery (ONW): 1,150 matches, 23 cells × 50 seeds, $276, 0 errors.**

Design: α fixed at ±4, budget spent on population structure. `(w,v)` = wolves × village-team seats steered, full 3×4 grid; seeds filtered to exactly 2 seated wolves; treated seats rotate with seed to avoid position confound. 0/1150 composition mismatches.

- **Spillover: null.** Untreated seats never move — 0/16 cells survive Bonferroni on vote accuracy, 0/8 on judged cooperation. **A steered minority does not drag the group**, up to 4 of 5 seats.
- **Direct effect: strong on judged behaviour, absent on actions.** Wolves at α=−4 drop **−10.7 judged cooperation points (p=3.5×10⁻⁸)** — while the same seats' vote accuracy doesn't move at all. This is the talk-vs-play gap *within one experiment*.
- **A preliminary 3-cell spillover (Δ=−0.21, p=0.012) did NOT replicate** on the full battery. It was an artifact of pooling only village-treated cells.
- **Win rate is the one place anything moves**, and it's the reverse of the obvious guess: steering *wolves* anti-cooperative doesn't help them (−0.05); degrading the *villagers* does (+0.197, p=0.024, monotone in dose 0.42→0.60→0.62→0.64). Anti-cooperative wolves become conspicuous and get caught. **Suggestive only** — 0/22 cells and 0/3 pooled contrasts survive correction, and Spearman trend p=0.037 vs Cochran-Armitage p=0.135.

**3. Not over-steering.** Blinded coherence audit (fluency only, conditions shuffled): α=−4 78.1, baseline 81.6, α=+4 79.5; 1/120 messages below 50, none below 25. The action-level null is not degeneration.

**Also:** per-seat and whole-table judging of the same construct diverge — whole-table is monotone in α, per-seat peaks at baseline and falls both ways. Re-anchoring the rubric to the paper's exact wording did not fix it, so it is the aggregation level. Treat per-seat α=+4 signs as instrument artifact until understood.

**Ops:** `update_autoscaler(max_containers=N)` changes concurrency on a *live* Modal deployment — no redeploy, so the 1,500-deployed-app workspace ceiling is irrelevant (I wrongly declared 80 impossible after a failed redeploy). SH matches average **48.8 min** vs ONW's 4.7; the launcher's 30-min default job timeout **silently skips** timed-out matches and would have discarded ~70% of a $2,921 SH run. Per-game timeouts now baked in. See [[project_judge_measurement_fragility]].

SH mixed-population battery (1,150 matches, ~$2,921, 160 concurrent) launched — results pending.

### 2026-07-28: Two-round judge overturns the gameplay-transfer claim; paper adjusted on deadline day [result]

Final act of the three-day measurement audit. Built the two-round instrument (user-specified design): **round 1** scores the whole table with the paper's exact cooperation wording (one scale; the original three were collinear), **round 2** scores each seat individually, named explicitly — both on complete untruncated god-view transcripts, blind to steering condition, round 2 never shown round 1's verdict. Re-judged everything (~15.6k Sonnet calls, ~$165, at `LLM_TOKENS_PER_MINUTE=2000000` — the 200k default is a self-imposed client throttle, not a provider limit).

**The published gameplay effect does not survive.** Same matches as the paper (`game_steering_v2`, `sh_steering_v2`):

| | published (talk-only, 9k cut) | R1 full transcript | R2 per-seat |
|---|---|---|---|
| ONW α=−4 | 42.4* | 44.0 (p=.80) | 53.0 (p=.45) |
| ONW baseline | 54.8 | 42.2 | 51.8 |
| ONW α=+4 | 63.6* | **33.4 (p=.0024)** | **43.5 (p=.0006)** |
| SH both arms | −4 significant* | null (p≥.45) | null (p≥.44) |

The pro-social contrast **reverses sign**. Progression: talk-only +8.8 → +actions +2.3 → full transcript **−8.8**. Independent replication: the 1,150-match ONW mixed battery shows the same reversal (R1 α=+4 p=0.0011, α=−4 null). Objective play (votes, win rates) null throughout; SH Liberal win 0.56→0.68→0.84 (extremes p=0.062) is the one suggestive objective trend.

**User's aggregation hypothesis confirmed at scale:** round 1 (table) sits below the mean of round 2's per-seat scores in **1,035/1,150 matches** (mean gap −12.5). Individually-reasonable players, collectively-uncooperative table — a pure aggregation-level effect, near-constant across conditions.

**Paper adjusted** (commit `f1a87f2`, submission day): gameplay claim withdrawn from abstract/intro/conclusion and reframed as a cautionary **measurement result** — "LLM-judged evaluation of agent collectives can manufacture effects from what the judge is shown." Original numbers retained in one paragraph labelled as the measurement under critique; Figure 5 caption marks non-robustness; Table 4 dropped; supplementary carries the two-round instrument + full results table. Behavioral/mechanistic/external-transfer layers untouched. 7+2pp verified.

**SH engine repair arc** (before the re-judge): first SH mixed battery voided 1,099/1,150 matches (baseline 100% — grammar, not steering). Fixes: self-inclusive whisper/alliance recipients now drop the sender; nomination prompt states eligibility positively; all metavariable prompts/errors (`<enact>I</enact>`, `<nominate>X</nominate>`) replaced with concrete tokens — models copy the metavariable literally then explain the assumption. Pilots: 4.4% → 20% → **80%** completion. Full battery re-run not yet approved (~$2,900).

**Session lesson, compressed:** every LLM-judged number this project published moved when the instrument changed — window (+8.8→+2.3), scales (3 collinear → 1), aggregation (whole-table vs per-seat, joint vs focused), and finally construct-on-fuller-input (sign reversal). The structural metrics and win rates never moved once. Judge-free measures first; judged measures only with an instrument-sensitivity check. Code: worktree commit `dee804b`. See [[project_judge_measurement_fragility]].

### 2026-07-28: The hidden-role games are the wrong instrument; building an intent/competence pair [design]

**Claim**: One Night Werewolf and Secret Hitler cannot answer the question the cooperation vector poses, because in a hidden-role game **cooperation is assigned, not chosen**. A Liberal cooperates because that is the win condition; a Fascist defects for the same reason. No seat ever faces the free cooperate-or-defect choice with both options available and both rational — which is exactly the structure of the PD vignettes the vector was fit on. What steering can move there is how well a seat executes its assigned team's script, plus talk style. That is a plausible reason the transfer looked weak even before the judging window turned out to be doing the work.

**Practical costs compound the structural problem** (Secret Hitler):

- ~174 turns/match, **48.8 min**, **$2.54/match**
- **20% abort rate**, and it is a long tail rather than one fixable bug — 10 aborts across **8 distinct signatures** in the n=50 pilot: invalid nominee (x3), malformed vote/nominate/say grammar (x4), dead whisper recipient, illegal execution target, self-whisper
- The aborts trace to a rich action surface (`whisper`/`nominate`/`execute`); lifting completion would need retry/legal-fallback changes to the engine, i.e. new behaviour, not a patch

**The replacement is a pair, not a game** — because one game cannot separate the two things that could produce a steering effect:

- **`public_goods`** — repeated N-player linear public goods with cheap talk and an optional Fehr-Gächter punishment stage. Free choice, material stakes, and a **judge-free dependent variable**: the number the seat submitted. Contribution is simultaneous, enforced at both masking points (prompts never show the current round; in-round contributions route to an empty audience with a god-log record). Measures cooperative **intent**.
- **`hanabi`** — the pure-cooperation control. Defection is not representable, everyone shares one payoff, so a steering effect here is coordination **competence**, not intent. Official 50-card deck, clue/fuse economy, strict-bomb scoring with the raw firework total preserved separately.

**Why the pair matters**: if the vector moves public-goods contributions *and* Hanabi score, it is a competence dial, not a cooperation setpoint — the same confound the MMLU/GSM8k/HumanEval controls rule out for single-vignette steering, but at the multi-agent level where no such control existed.

**Expected operating characteristics** (to be checked against a pilot, not trusted):

| | SH | public goods | Hanabi |
|---|---|---|---|
| min/match | 48.8 | ~10 | ~4 |
| $/match | 2.54 | ~0.55 | ~0.25 |
| abort rate | 20% | ~1% (target) | ~1% (target) |

Both have tiny closed action grammars (`<contribute>N</contribute>`, `<play>N</play>`, `<clue>P2 red</clue>`), which is where the abort-rate gain is expected to come from.

**Design axis**: role-free games have no team asymmetry to cross, so the (w, v) role-composition grid collapses to the dose axis alone — k = 0..5 steered seats at each sign, 11 cells. Recorded as (w=0, v=k) so the manifest schema is unchanged.

**Status**: implemented, tested (35 new tests; 350 in `tests/play`, 676 overall), committed as `b299a88`. Deploying to Modal is deliberately deferred until the SH battery finishes — a redeploy mid-run would risk ~$2,900 of in-flight work.

**Also today**: the SH verification gate passed (9/12 completed, 0 treated-seat echo mismatches, no new abort signatures) and the 1,150-match mixed-population battery launched at the deployed 160-container cap (~5.8h, ~$2,900). Its purpose is the **judge-free** measure — win rates — for the game whose judged result we withdrew from the paper this morning. Also fixed a launcher bug that wrote `winner: null` for every resumed log, marking completed matches as unfinished (`scripts/repair_manifest_winners.py` re-derives them from the logs).

**Open question worth stating plainly**: if the game is the wrong instrument, clean win rates from it are still a weak artifact. The battery is worth running — it completes an honest dataset and executes the mixed-population design — but the next cycle's money is better spent on the pair above.